# 01 · Multi-Agent Architectures — 动手 Demo

> **阅读方式**：本 Notebook 采用「先概念、后实操」的组织方式 —— 每个 Part 先用 `📖 概念` 讲清设计动机与架构原理，再用 `🧪 实操` 跑一段可独立运行的代码，最后给出 `关键观察`。

## 学习目标

学完本课，你能够：

1. **设计**分层多 Agent 架构（Worker / Orchestrator / Supervisor）
2. **选择**合适的协作模式（Pipeline、Hub-and-Spoke、Blackboard、Orchestrator-Driven、P2P）
3. **实现** Agent Supervision：Trace、质量门、HITL、审计
4. **掌握**工程要点：通信、幂等、可观测、降级、安全
5. **完成**一个包含 Orchestrator + Verifier + Supervisor 的本地演示系统

## 📑 目录

> 约定：`📖 概念` = 讲义幻灯片；`🧪 实操` = 可运行的动手代码。

| 章节 | 主题 | 内容 |
|------|------|------|
| **0** | 环境准备 | 🧪 模拟 LLM / 向量检索工具 |
| **Part 1** | 单体 Agent 的局限 | 📖 为什么需要多 Agent · 🧪 单体 Agent |
| **Part 2** | 设计原则与架构模式 | 📖 三种基础模式 + Map-Reduce + Orchestrator 驱动 + P2P + 选择决策树 |
| **Part 3** | Pipeline 模式 | 🧪 队列 + Worker |
| **Part 4** | Hub-and-Spoke 模式 | 🧪 asyncio 并行编排 |
| **Part 5** | Blackboard 模式 | 🧪 共享状态 + 事件 |
| **Part 6** | 通信机制与状态管理 | 📖 通信机制 · 状态与记忆 |
| **Part 7** | 幂等设计 | 📖 概念 · 生产级实现与 UPSERT · 重试退避 · 🧪 幂等处理器 |
| **Part 8** | Backpressure 与流量控制 | 📖 背压 · 🧪 有界队列 |
| **Part 9** | 可观测性 | 📖 三大支柱 · 🧪 Metrics + Trace |
| **Part 10** | 安全隔离与最小权限 | 📖 安全清单 · 🧪 输入校验 + 权限矩阵 |
| **Part 11** | 成本优化策略 | 📖 路由 / 缓存 / 批处理 |
| **Part 12** | 优雅降级 | 🧪 降级路径 |
| **Part 13** | 进阶：三个完整实现 | 📖 Redis / HTTP / Blackboard |
| **Part 14** | 分层架构：Worker / Orchestrator / Supervisor | 📖 三层职责与权衡 · 🧪 三层实现 + TraceStore |
| **Part 15** | Agent Supervision | 📖 Trace 设计 / Quality Gate / HITL / Audit · 🧪 Verifier + 质量门 + HITL |
| **Part 16** | 实战：三页报告系统 | 📖 逐层拆解 · 🧪 端到端运行 |
| **Part 17** | 最佳实践与常见坑 | 📖 Top 10 实践 + 坑速查 |
| **Part 18** | 课堂练习 | 📖 五阶段路径 · 评估标准 · 演示顺序 |
| **总结** | 小结 · 资源 · Q&A | — |

## 0 · 环境准备

先加载依赖，并定义后续所有 Demo 都会用到的**模拟工具**（检索 / 重排 / 生成 / 校验）。

In [1]:
import asyncio
import copy
import queue
import random
import threading
import time
import uuid
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Any

import nest_asyncio

nest_asyncio.apply()

# ── 模拟 LLM / 向量检索 ─────────────────────────────────────────────────────
CORPUS = [
    "RAG 通过检索外部文档来扩充 LLM 的上下文。",
    "向量数据库支持语义相似度搜索。",
    "Reranker 用交叉编码器对候选文档重新打分。",
    "幂等设计让系统在重试时不产生副作用。",
    "Backpressure 防止下游服务因流量过大而崩溃。",
    "可观测性包括 Metrics、Traces 和 Logs 三大支柱。",
    "Graceful degradation 让系统在部分故障时仍能返回有用结果。",
    "Hub-and-Spoke 模式由 Orchestrator 统一分派任务。",
    "Blackboard 模式通过共享工作区实现多 Agent 协作。",
    "消息队列解耦生产者和消费者，支持异步处理。",
]

def fake_retrieve(query: str, top_k: int = 5, latency: float = 0.3) -> list[dict]:
    """模拟向量检索，返回 top_k 相关文档"""
    time.sleep(latency)
    scored = [(doc, random.uniform(0.5, 1.0)) for doc in CORPUS]
    scored.sort(key=lambda x: -x[1])
    return [{"text": doc, "score": round(s, 3)} for doc, s in scored[:top_k]]

def fake_rerank(query: str, hits: list[dict], top_k: int = 3, latency: float = 0.2) -> list[dict]:
    """模拟 cross-encoder reranker"""
    time.sleep(latency)
    reranked = copy.deepcopy(hits)
    for h in reranked:
        h["score"] = round(h["score"] * random.uniform(0.9, 1.1), 3)
    reranked.sort(key=lambda x: -x["score"])
    return reranked[:top_k]

def fake_generate(question: str, context: list[dict], latency: float = 0.5) -> str:
    """模拟 LLM 生成"""
    time.sleep(latency)
    snippets = " ".join(h["text"][:30] for h in context[:2])
    return f"[答案] 关于「{question[:20]}」：{snippets}..."

def fake_verify(answer: str, context: list[dict], latency: float = 0.15) -> dict:
    """模拟答案校验"""
    time.sleep(latency)
    return {"grounded": random.random() > 0.2, "confidence": round(random.uniform(0.7, 1.0), 2)}

print("✅ 模拟工具加载完毕")

✅ 模拟工具加载完毕


## Part 1 · 单体 Agent 的局限

### 📖 概念 · 引言：为什么需要多 Agent

#### 单体 Agent 的局限

```
用户 ──▶ [ 一个 LLM + 若干工具 ] ──▶ 回答
```

**在以下场景会遇到瓶颈：**

- **规模瓶颈**：检索 × 生成 × 验证串行，延迟叠加
- **职责模糊**：一个 Agent 承担过多角色，难以优化
- **无法独立扩容**：检索并发高但生成成本高，无法分别调配
- **故障扩散**：任一环节失败导致整体不可用
- **迭代困难**：换一个 Reranker 需要改动整个 Agent

---

#### 多 Agent 带来什么？

```
用户
 │
 ▼
Orchestrator
 ├──▶ Retriever  (高并发，独立扩容)
 ├──▶ Reranker   (可替换实现)
 ├──▶ Generator  (成本精细控制)
 └──▶ Verifier   (独立故障隔离)
```

- **职责分离** — 每个 Agent 专注一件事，易测试、易替换
- **独立扩容** — 各服务按需弹性，成本可控
- **故障隔离** — Reranker 挂了可降级，不影响其他路径
- **并行执行** — 多个 Agent 同时工作，总延迟更短

---

### 🧪 实操：单体 Agent（串行执行）

所有逻辑塞在一个函数里：检索 → 重排 → 生成 → 校验，**串行执行**。

```
用户 ──▶ [ monolithic_agent() ] ──▶ 回答
            ├─ retrieve  (0.3s)
            ├─ rerank    (0.2s)
            ├─ generate  (0.5s)
            └─ verify    (0.15s)
               ──────────────────
               总计 ≈ 1.15s（串行叠加）
```

In [2]:
def monolithic_agent(query: str) -> dict:
    """单体 Agent：所有步骤串行执行"""
    t0 = time.perf_counter()

    hits    = fake_retrieve(query, top_k=5)
    topk    = fake_rerank(query, hits, top_k=3)
    answer  = fake_generate(query, topk)
    verdict = fake_verify(answer, topk)

    elapsed = round(time.perf_counter() - t0, 3)
    return {"answer": answer, "verdict": verdict, "latency_s": elapsed}

# ── 运行 ─────────────────────────────────────────────────────────────────────
result = monolithic_agent("什么是 RAG？")
print(f"延迟：{result['latency_s']}s")
print(f"校验：{result['verdict']}")
print(f"答案：{result['answer'][:80]}...")

延迟：1.169s
校验：{'grounded': True, 'confidence': 0.73}
答案：[答案] 关于「什么是 RAG？」：Graceful degradation 让系统在部分故障时 向量数据库支持语义相似度搜索。......


> **观察**：延迟 ≈ 各步骤之和。任何一步慢，整体就慢。
> 接下来我们拆分为多 Agent，观察延迟和可维护性的变化。

## Part 2 · 设计原则与架构模式

### 多 Agent 的四大设计原则

<div class="columns">
<div>

#### 1. 职责分离 SRP
每个 Agent 只做一件明确的事
- Retriever → 只负责检索
- Reranker  → 只负责重排
- Generator → 只负责生成
- Verifier  → 只负责校验

</div>
<div>

#### 2. 松耦合通信
- 通过消息/队列/API 交换数据
- 不直接调用对方内部函数
- 接口契约（schema）稳定

#### 3. 可独立扩展
- 每个 Agent 独立部署
- 按负载单独扩缩容

#### 4. 可替换性
- 换掉一个 Agent 不影响其他

</div>
</div>

---
### 架构模式

#### 架构模式 1：Pipeline（流水线）

```
请求
 │
 ▼
[Retriever] ──▶ [Reranker] ──▶ [Generator] ──▶ [Verifier] ──▶ 响应
```

**特点**
- 线性、顺序处理，易于理解与调试
- 每个阶段输出即下一阶段输入
- 适合：文档 QA、内容审核、数据清洗流水线

**实现方式**
- 消息队列（Redis / Kafka）：每个 Worker 消费上游队列，写入下游队列
- 同步链式调用：Orchestrator 依次调用各服务

---

#### 架构模式 2：Hub-and-Spoke（协调器，轮毂-辐条模式）

```
                ┌─── Retriever ──┐
                │                │
用户 ──▶ Orchestrator ─── Reranker ───▶ 合成回复
                │                │
                └─── Generator ──┘
```

**特点**
- Orchestrator 负责任务分派与结果合成
- 支持并行调用多个 Agent
- 适合：复杂问答、多步推理、工作流编排

**注意**
- Orchestrator 成为单点，需做好容错与限流
- 建议用异步 + 回调，避免阻塞

---

#### 架构模式 3：Blackboard（共享工作区）

```
          写入              读取
Agent A ──────▶ Blackboard ◀────── Agent C
Agent B ──────▶     │     ◀────── Agent D
                    │
              变更通知（事件）
```

**特点**
- 多个 Agent 共享同一"黑板"（数据库/缓存/对象存储）
- 通过事件/轮询感知变化，异步推进
- 适合：迭代协作、多模型协商、长任务

**挑战**
- 需要设计好写入合约与冲突处理
- 数据清理策略（TTL / GC）

---

### 三种模式对比

| 维度 | Pipeline | Hub-and-Spoke | Blackboard |
|------|----------|---------------|------------|
| 复杂度 | 低 | 中 | 高 |
| 调试难度 | 易 | 中 | 难 |
| 并行能力 | 低 | 高 | 高 |
| 适合任务 | 线性 | 多角色协作 | 迭代协商 |
| 典型实现 | Redis Queue | FastAPI + gRPC | DB + Pub-Sub |

> **建议**：从 Pipeline 起步，按需演进到 Hub-and-Spoke，Blackboard 用于高级场景

---

### 协调策略

<div class="columns">
<div>

#### 中心化编排
```
Orchestrator
 ├─ 分配任务
 ├─ 控制顺序
 └─ 合成结果
```
✅ 可控、易监控、易审计

❌ 单点复杂性、扩展受限

</div>
<div>

#### 去中心化 / 涌现式
```
Agent A ◀──▶ Agent B
  ▲               ▲
  └───▶ Agent C ──┘
```
✅ 高鲁棒性、天然分布式

❌ 设计复杂、调试困难

#### Hybrid（推荐）
中心化编排 + 局部自治
平衡可控性与弹性

</div>
</div>

---

### 架构模式补充：Orchestrator-Driven、Peer-to-Peer、模式选择

#### 架构模式 4：Orchestrator-Driven（中心编排，主力模式）

Orchestrator-Driven 是 Hub-and-Spoke 的强化版：Orchestrator 不仅并行分派，还负责**生成计划**、**聚合结果**、**调用校验**、**上报监督**。

```mermaid
flowchart TB
    User[用户] --> Orch[Orchestrator]
    Orch --> Plan["1. plan → summarize / extract_risks / propose_measures"]
    Plan --> D1["2. dispatch summarize"]
    Plan --> D2["2. dispatch extract_risks"]
    Plan --> D3["2. dispatch propose"]
    D1 --> SW[SummarizerWorker]
    D2 --> RW[RiskExtractorWorker]
    D3 --> MW[MeasureProposer]
    SW --> Agg["3. aggregate → report"]
    RW --> Agg
    MW --> Agg
    Agg --> Ver["4. verify → VerifierAgent → verdict"]
    Ver --> Sup["5. report_to_supervisor"]
```

**优势**：全程可控、每步可追踪、易加入质量门
**关键设计点**：计划可以**静态**（hardcoded）或**动态**（LLM 生成）

##### 静态 vs 动态 Orchestrator

```python
# 静态：预定义计划，固定工作流
def plan(doc_id):
    return [
        {"task": "summarize",     "depends_on": []},
        {"task": "extract_risks", "depends_on": []},
        {"task": "propose",       "depends_on": ["extract_risks"]},
        {"task": "verify",        "depends_on": ["summarize", "propose"]},
    ]
```

| | 静态 Orchestrator | 动态 Orchestrator（ReAct / LangGraph） |
|---|---|---|
| 计划来源 | 预定义任务列表 | LLM 生成 `TaskListSchema` |
| 可预测性 | ✅ 高 | ❌ 低 |
| 成本 | ✅ 低 | ❌ 高（额外 LLM 调用） |
| 调试 | ✅ 易 | ❌ 难 |
| 适用 | 固定流程业务（覆盖 80% 场景） | 开放式 Agentic 系统（剩余 20%） |

> **生产建议**：先用静态 Orchestrator 覆盖已知流程，对确实需要自适应的场景再引入动态计划。

#### 架构模式 5：Peer-to-Peer（去中心化）

```mermaid
flowchart TB
    A[Agent A] <-->|negotiate| B[Agent B]
    A <-->|bid / accept| C[Agent C]
    B <-->|bid / accept| C
```

**典型场景**
- 多个 Generator 竞价，Orchestrator 选最优结果
- 资源分配：多 Agent 协商共享 GPU / API 配额
- 多专家辩论：不同角色 Agent 对同一问题发表意见

**实现**：通常是 Blackboard + 投票 / 竞争规则

> ⚠️ **生产慎用**：调试极难、行为难预测、成本不可控
> ✅ **研究价值高**：适合 AI 自治系统 / 多智能体仿真

#### 模式选择决策树

```mermaid
flowchart TB
    Q1{"任务是否有固定线性阶段？"}
    Q1 -->|YES| P[Pipeline]
    Q1 -->|NO| Q2{"是否并行处理多个相似子任务？"}
    Q2 -->|YES| MR[Parallel / Map-Reduce]
    Q2 -->|NO| Q3{"是否需要复杂计划与自适应？"}
    Q3 -->|YES| OD[Orchestrator-Driven 静态 or 动态]
    Q3 -->|NO| Q4{"是否需要多 Agent 迭代共享中间产物？"}
    Q4 -->|YES| BB[Blackboard]
    Q4 -->|NO| Q5{"是否是自治系统 / 多方协商？"}
    Q5 -->|YES| P2P[Peer-to-Peer]
    Q5 -->|NO| Single[重新检查需求，可能单 Agent 就够了]
```

> **选择建议**：从 Pipeline 起步 → 按需引入 Parallel → Orchestrator-Driven 是主力模式 → Blackboard 用于复杂迭代 → P2P 留给研究。

### 架构模式补充：Parallel / Map-Reduce（并行 + 汇总）

```mermaid
flowchart TB
    Doc[文档] -->|split into chunks| Split
    Split --> S1[Summarizer-1]
    Split --> S2[Summarizer-2]
    Split --> S3[Summarizer-3]
    Split --> RE[RiskExtractor]
    S1 --> Agg[Aggregator]
    S2 --> Agg
    S3 --> Agg
    RE --> Agg
    Agg --> Final[final_report]
```

**关键点**
- **Map 阶段并行**：把输入切分给多个同类 Worker
- **Reduce 阶段汇总**：Aggregator 合并、去重、排序
- 总延迟 ≈ `max(子任务延迟)`，而非 `sum`
- 特别适合：多文档摘要、批量实体提取

**Reduce 阶段必须处理部分失败**，常见策略：

| 策略 | 行为 | 适用 |
|------|------|------|
| Fail-fast | 任一 Map 失败即整体失败 | 强一致性要求 |
| Best-effort | 跳过失败分片，标注缺失 | 摘要 / 检索类 |
| Retry-then-degrade | 重试失败分片，仍失败则降级 | 大多数生产场景 |

> Part 4 的 Hub-and-Spoke Demo 就是 Map-Reduce 的简化版：`asyncio.gather` 负责 Map，Orchestrator 负责 Reduce。

## Part 3 · Pipeline 模式（`queue.Queue` + 线程 Worker）

用 Python 内置 `queue.Queue` 模拟 Redis 列表，每个 Worker 是一个线程。

```
Orchestrator
     │ put(job)
     ▼
q_retrieve ──▶ RetrieverWorker
                    │ put(job+hits)
                    ▼
              q_rerank ──▶ RerankerWorker
                                │ put(job+topk)
                                ▼
                          q_generate ──▶ GeneratorWorker
                                              │ put(result)
                                              ▼
                                         results[job_id]
```

In [3]:
# ── 队列定义 ─────────────────────────────────────────────────────────────────
q_retrieve = queue.Queue()
q_rerank   = queue.Queue()
q_generate = queue.Queue()
results    = {}          # job_id → result（生产中用 Redis / DB）
_stop      = threading.Event() # 停止信号

# ── Worker 定义 ───────────────────────────────────────────────────────────────
def retriever_worker():
    while not _stop.is_set(): # 没收到停止信号就一直循环
        try:
            job = q_retrieve.get(timeout=0.2)
        except queue.Empty:
            continue # 没任务, 回去检查 _stop
        print(f"  [Retriever] processing {job['request_id'][:8]}...")
        job = {**job, "hits": fake_retrieve(job["query"], latency=0.3)}
        q_rerank.put(job) # 放到下一阶段的队列中
        q_retrieve.task_done() # 告诉队列“这个任务完成了”

def reranker_worker():
    while not _stop.is_set():
        try:
            job = q_rerank.get(timeout=0.2)
        except queue.Empty:
            continue
        print(f"  [Reranker]  processing {job['request_id'][:8]}...")
        job = {**job, "topk": fake_rerank(job["query"], job["hits"], latency=0.2)}
        q_generate.put(job)
        q_rerank.task_done()

def generator_worker():
    while not _stop.is_set():
        try:
            job = q_generate.get(timeout=0.2)
        except queue.Empty:
            continue
        print(f"  [Generator] processing {job['request_id'][:8]}...")
        answer  = fake_generate(job["query"], job["topk"], latency=0.5)
        verdict = fake_verify(answer, job["topk"], latency=0.15)
        results[job["request_id"]] = {
            "answer": answer,
            "verdict": verdict,
            "finished_at": time.perf_counter(),
        }
        q_generate.task_done()

# ── 启动 Workers ──────────────────────────────────────────────────────────────
_stop.clear() # 确保标志是"未停止"状态（重复运行 notebook 时有用）
workers = [
    threading.Thread(target=retriever_worker, daemon=True), # daemon=True：主线程退出时它们会自动结束
    threading.Thread(target=reranker_worker,  daemon=True),
    threading.Thread(target=generator_worker, daemon=True),
]
for w in workers:
    w.start()

print("✅ 3 个 Worker 已启动（Retriever / Reranker / Generator）")

✅ 3 个 Worker 已启动（Retriever / Reranker / Generator）


In [4]:
# ── 提交多个查询，观察并发流水线 ────────────────────────────────────────────
queries = ["什么是 RAG？", "幂等设计是什么？", "如何实现可观测性？"]
job_ids = []
t_start = time.perf_counter()

for q in queries:
    job_id = str(uuid.uuid4())
    job_ids.append((job_id, q, time.perf_counter()))
    q_retrieve.put({"request_id": job_id, "query": q})
    print(f"➡ 提交：{q[:20]}  [{job_id[:8]}]")

# 等待所有结果
print("\n⏳ 等待结果...")
for job_id, query, t_submit in job_ids:
    while job_id not in results:
        time.sleep(0.05)
    r = results[job_id]
    latency = round(r["finished_at"] - t_submit, 3)
    print(f"  ✅ [{job_id[:8]}] {query[:20]} → {latency}s | grounded={r['verdict']['grounded']}")

total = round(time.perf_counter() - t_start, 3)
print(f"\n总耗时：{total}s（3 个查询并行流水线）")

# 停止 Workers
_stop.set()

for w in workers:
    w.join(timeout=2)         # 等线程真正退出，最多等 2 秒

print([w.is_alive() for w in workers])   # 全是 False 说明都退出了

➡ 提交：什么是 RAG？  [a40a6a58]  [Retriever] processing a40a6a58...

➡ 提交：幂等设计是什么？  [a04a2ed4]
➡ 提交：如何实现可观测性？  [9f15c249]

⏳ 等待结果...
  [Retriever] processing a04a2ed4...  [Reranker]  processing a40a6a58...

  [Generator] processing a40a6a58...
  [Retriever] processing 9f15c249...
  [Reranker]  processing a04a2ed4...
  [Reranker]  processing 9f15c249...
  [Generator] processing a04a2ed4...
  ✅ [a40a6a58] 什么是 RAG？ → 1.171s | grounded=False
  [Generator] processing 9f15c249...
  ✅ [a04a2ed4] 幂等设计是什么？ → 1.831s | grounded=True
  ✅ [9f15c249] 如何实现可观测性？ → 2.49s | grounded=True

总耗时：2.51s（3 个查询并行流水线）


### Pipeline 执行时序图
#### 用甘特图看"谁在什么时间段占用哪个 worker"
```mermaid
gantt
    title 3 个查询的流水线执行时间线（单位：秒）
    dateFormat x
    axisFormat %S.%L

    section Retriever
    A 检索 (a40a6a58) :a1, 0, 300
    B 检索 (a04a2ed4) :b1, 300, 600
    C 检索 (9f15c249) :c1, 600, 900

    section Reranker
    A 重排 :a2, 300, 500
    B 重排 :b2, 600, 800
    C 重排 :c2, 900, 1100

    section Generator
    A 生成+校验 :crit, a3, 500, 1150
    B 生成+校验 :crit, b3, 1150, 1800
    C 生成+校验 :crit, c3, 1800, 2450
```

#### 用时序图展示任务在各队列和 worker 之间的流转
```mermaid
sequenceDiagram
    autonumber
    participant M as 主线程
    participant R as Retriever (0.3s)
    participant K as Reranker (0.2s)
    participant G as Generator (0.65s)

    M->>R: 提交 A、B、C（几乎同时）
    Note over M: 轮询 results 等待

    R->>K: A 检索完成 (t=0.3)
    activate K
    R->>K: B 检索完成 (t=0.6)
    K->>G: A 重排完成 (t=0.5)
    activate G
    R->>K: C 检索完成 (t=0.9)
    K->>G: B 重排完成 (t=0.8)<br/>进入 q_generate 排队
    K->>G: C 重排完成 (t=1.1)<br/>进入 q_generate 排队
    deactivate K

    G-->>M: A 完成 (t≈1.17s)
    Note over G: B 开始生成
    G-->>M: B 完成 (t≈1.83s)
    Note over G: C 开始生成
    G-->>M: C 完成 (t≈2.49s)
    deactivate G

    M->>M: _stop.set() 停止所有 worker
```

#### 从图中能看出什么
1. **前两段是流水线并行的**：A 在重排时，B 已经在检索；Retriever 每 0.3 秒吐出一个任务，节奏很均匀。
2. **Generator 是瓶颈**：它每个任务要 0.65 秒，比上游的 0.3 秒慢得多。B 和 C 重排早就完成了（0.8s、1.1s），却只能在 `q_generate` 里排队，B 要等到 1.15s、C 要等到 1.8s 才轮到。
3. **完成时间间隔固定为约 0.65s**（1.17 → 1.83 → 2.49），正好等于瓶颈阶段的单任务耗时，这就是前面说的"吞吐量由最慢阶段决定"。
4. **总耗时 2.51s**，对比完全串行的 3 × 1.15 = 3.45s，只快了约 27%。因为 Generator 是瓶颈，上游的并行优势被浪费了。如果开 2 个 Generator 线程，B 和 C 的生成可以重叠，总耗时大约能降到 1.8s 左右。

> **关键观察**：
> - 三个查询**同时**在流水线中流动，总时间 ≈ 单次时间（而非 3×）
> - 每个 Worker 职责单一，替换 Reranker 不影响其他两个
> - 队列天然起到**背压缓冲**作用

## Part 4 · Hub-and-Spoke 模式（轮毂-辐条模式，asyncio 并行）

Orchestrator 用 `asyncio.gather` **并行**调用多个 Agent，再合成结果。

```
用户
 │
 ▼
Orchestrator
 ├─── asyncio.gather ──▶ RetrieverAgent  (0.3s)
 │                  ──▶ MetadataAgent   (0.1s)  ← 同时运行！
 │                  ──▶ SummaryAgent    (0.2s)
 │         ▼ 全部完成后
 └─── GeneratorAgent（拿到所有结果再生成）
```

In [5]:
# ── 异步 Agent 定义 ──────────────────────────────────────────────────────────
async def async_retrieve(query: str) -> list[dict]:
    await asyncio.sleep(0.3)   # 模拟 I/O 延迟
    return fake_retrieve(query, top_k=5, latency=0)

async def async_metadata(query: str) -> dict:
    """模拟获取文档元数据"""
    await asyncio.sleep(0.1)
    return {"doc_count": random.randint(100, 500), "last_updated": "2025-01"}

async def async_summary_hint(query: str) -> str:
    """模拟摘要提示词生成"""
    await asyncio.sleep(0.2)
    return f"请用简洁语言回答：{query}"

async def orchestrator(query: str) -> dict:
    t0 = time.perf_counter()
    print(f"  [Orchestrator] 并行调用 3 个 Agent...")

    # ── 并行阶段：同时获取检索结果、元数据、提示词 ──────────────────────────
    hits, meta, hint = await asyncio.gather(
        async_retrieve(query),
        async_metadata(query),
        async_summary_hint(query),
    )

    print(f"  [Orchestrator] 并行完成，开始生成...")

    # ── 串行阶段：依赖上面结果的生成步骤 ────────────────────────────────────
    await asyncio.sleep(0.5)   # 模拟生成
    answer = fake_generate(hint, hits, latency=0)

    elapsed = round(time.perf_counter() - t0, 3)
    return {
        "answer": answer,
        "meta": meta,
        "latency_s": elapsed,
        "parallel_savings_s": round((0.3 + 0.1 + 0.2) - max(0.3, 0.1, 0.2), 3),
    }

# ── 运行 ─────────────────────────────────────────────────────────────────────
result = asyncio.run(orchestrator("什么是向量数据库？"))
print(f"\n总延迟：{result['latency_s']}s")
print(f"并行节省：{result['parallel_savings_s']}s（相比串行）")
print(f"文档数：{result['meta']['doc_count']}")

  [Orchestrator] 并行调用 3 个 Agent...
  [Orchestrator] 并行完成，开始生成...

总延迟：0.81s
并行节省：0.3s（相比串行）
文档数：107


> **关键观察**：
> - 3 个 Agent 并行，总时间 ≈ `max(0.3, 0.1, 0.2) + 0.5 = 0.8s`
> - 串行则需 `0.3 + 0.1 + 0.2 + 0.5 = 1.1s`
> - `asyncio.gather` 是实现并行的关键

## Part 5 · Blackboard 模式（共享状态 + 事件）

多个 Agent 通过**共享黑板**（字典 + `threading.Event`）协作，无需直接互调。

```
                  写入         订阅事件
IngestAgent   ──▶ bb["chunks"] → event: chunks_ready
RetrieverAgent ──▶ bb["hits"]  → event: hits_ready
RerankerAgent  ──▶ bb["topk"]  → event: topk_ready
GeneratorAgent ──▶ bb["answer"]→ event: done
```

In [6]:
# ── Blackboard 实现 ──────────────────────────────────────────────────────────
class Blackboard:
    def __init__(self):
        self._data: dict[str, Any] = {}
        self._events: dict[str, threading.Event] = defaultdict(threading.Event)
        self._lock = threading.Lock()

    def write(self, key: str, value: Any):
        with self._lock:
            self._data[key] = value
        print(f"  📋 Blackboard.write({key!r})")
        self._events[key].set()          # 通知订阅者

    def read(self, key: str) -> Any:
        with self._lock:
            return self._data.get(key)

    def wait_for(self, key: str, timeout: float = 5.0) -> bool:
        """阻塞直到 key 被写入"""
        return self._events[key].wait(timeout=timeout)


# ── 各 Agent 定义（每个都是独立线程）────────────────────────────────────────
def ingest_agent(bb: Blackboard, query: str):
    print("  [Ingest] 开始分片...")
    time.sleep(0.1)
    bb.write("query", query)
    bb.write("chunks", [{"text": t} for t in CORPUS[:4]])

def retriever_agent(bb: Blackboard):
    bb.wait_for("chunks")
    query  = bb.read("query")
    chunks = bb.read("chunks")
    print("  [Retriever] 检索中...")
    time.sleep(0.3)
    hits = [
        {"text": c["text"], "score": round(random.uniform(0.5, 1.0), 3)}
        for c in sorted(chunks, key=lambda _: random.random())[:3]
    ]
    bb.write("hits", hits)

def reranker_agent(bb: Blackboard):
    bb.wait_for("hits")
    query = bb.read("query")
    hits  = bb.read("hits")
    print("  [Reranker] 重排中...")
    time.sleep(0.2)
    topk = fake_rerank(query, hits, top_k=2, latency=0)
    bb.write("topk", topk)

def generator_agent(bb: Blackboard):
    ok = bb.wait_for("topk", timeout=5.0)
    query = bb.read("query")
    topk  = bb.read("topk") or []
    print("  [Generator] 生成中...")
    answer = fake_generate(query, topk, latency=0.5)
    bb.write("answer", answer)


# ── 运行：启动所有 Agent，让它们通过黑板协作 ─────────────────────────────────
bb = Blackboard()
t0 = time.perf_counter()

threads = [
    threading.Thread(target=ingest_agent,    args=(bb, "什么是 Blackboard 模式？")),
    threading.Thread(target=retriever_agent, args=(bb,)),
    threading.Thread(target=reranker_agent,  args=(bb,)),
    threading.Thread(target=generator_agent, args=(bb,)),
]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"\n⏱ 总耗时：{round(time.perf_counter()-t0, 3)}s")
print(f"📝 最终答案：{bb.read('answer')[:80]}...")

  [Ingest] 开始分片...
  📋 Blackboard.write('query')
  📋 Blackboard.write('chunks')
  [Retriever] 检索中...
  📋 Blackboard.write('hits')
  [Reranker] 重排中...
  📋 Blackboard.write('topk')
  [Generator] 生成中...
  📋 Blackboard.write('answer')

⏱ 总耗时：1.112s
📝 最终答案：[答案] 关于「什么是 Blackboard 模式？」：幂等设计让系统在重试时不产生副作用。 向量数据库支持语义相似度搜索。......


> **关键观察**：
> - 各 Agent **不互相调用**，只通过黑板读写
> - `wait_for` 实现事件驱动，无需轮询
> - 新增一个 Agent 只需监听相应字段，**不修改任何已有代码**

## Part 6 · 通信机制与状态管理

### 6.1 四种通信机制

| 机制 | 代表技术 | 延迟 | 耦合度 | 适用场景 |
|------|----------|------|--------|----------|
| **Sync RPC** | HTTP / gRPC | 低 | 高 | 实时决策、简单编排 |
| **Async Queue** | Redis / Kafka / SQS | 中 | 低 | 耗时操作、高并发 |
| **Pub-Sub** | SNS / Kafka Topic | 中 | 很低 | 广播事件、多订阅方 |
| **Shared Storage** | S3 + DB + 信号 | 高 | 很低 | 大文件、artifacts 共享 |


### 6.2 选择通信机制的决策树

```
需要立即拿到结果？
 │
 ├─ YES → 延迟是否 < 100ms？
 │          ├─ YES → gRPC / HTTP
 │          └─ NO  → HTTP + polling / WebSocket
 │
 └─ NO → 任务是否耗时 > 1s？
           ├─ YES → Message Queue（Redis / Kafka）
           └─ NO  → 可以用 async HTTP + callback
```

**黄金法则**
- 用户直接等待 → 同步 RPC
- 后台批处理 → 异步队列
- 广播通知 → Pub-Sub
- 大文件传递 → Shared Storage


### 6.3 状态与记忆

#### Agent State 分类

- **短期状态（任务内）**
    - 随消息一起传递
    - 无需持久化
    - 示例：当前检索结果

- **长期状态（跨会话）**
    - 存 DB / Redis
    - 需要 TTL 和生命周期管理
    - 示例：用户偏好、对话历史

## Part 7 · 幂等设计（Idempotency）

**什么是幂等（Idempotency）？** 同一个操作执行 1 次和执行 N 次，对系统产生的**效果完全相同**（数学上即 $f(f(x)) = f(x)$）。

| 类型 | 典型示例 | 重复执行时 |
|------|----------|------------|
| ✅ 幂等 | `SET k v`、`DELETE id=5`、`GET` | 结果不变，可安全重试 |
| ❌ 非幂等 | `count += 1`、`INSERT`（无唯一键）、`POST /pay` 扣款 | 产生副作用（如重复扣款） |

**为什么必须幂等？** 消息队列在网络超时、消费者崩溃恢复时会**重复投递**同一条消息；若处理逻辑不幂等，就会导致重复扣款 / 重复下单 / 脏数据 / 多余的 API 调用。

**怎么实现？** 给每条消息附一个全局唯一的 `request_id` 作为**幂等键**，处理前先查是否已处理过：

```python
# 每条消息带唯一 ID
job = {
    "request_id": "uuid-xxx",
    "query": "...",
}

# 处理前检查是否已处理
if redis.get(f"done:{job['request_id']}"):
    return  # 跳过，防止重复

process(job)
redis.setex(f"done:{job['request_id']}", 3600, 1)
```

### 🧪 实操：幂等设计（`request_id` + 已处理集合）

**问题**：消息队列在网络故障或消费者崩溃后会**重复投递**，如果处理不是幂等的，就会产生重复写入或重复 API 调用。

**解决方案**：每条消息带 `request_id`，处理前检查是否已处理过。

In [7]:
# ── 模拟幂等处理器 ───────────────────────────────────────────────────────────
class IdempotentProcessor:
    def __init__(self):
        self._processed: set[str] = set()    # 生产中用 Redis SET + TTL
        self._call_count = 0

    def process(self, job: dict) -> dict | None:
        rid = job["request_id"]

        # 幂等检查
        if rid in self._processed:
            print(f"  ⚠️  [{rid[:8]}] 重复消息，跳过（已处理过）")
            return None

        self._call_count += 1
        self._processed.add(rid)
        result = fake_generate(job["query"], fake_retrieve(job["query"], latency=0), latency=0)
        print(f"  ✅ [{rid[:8]}] 处理完成（第 {self._call_count} 次实际处理）")
        return {"result": result}


processor = IdempotentProcessor()

# ── 模拟场景：同一条消息被投递 3 次（网络重试）───────────────────────────────
job = {"request_id": str(uuid.uuid4()), "query": "幂等性是什么？"}

print("=== 模拟消息重复投递 ===")
for i in range(3):
    print(f"\n投递第 {i+1} 次：")
    processor.process(job)

print(f"\n实际处理次数：{processor._call_count}（应为 1，不是 3）")

# ── 不同 request_id 的新消息正常处理 ─────────────────────────────────────────
print("\n=== 新消息（不同 request_id）===")
for q in ["什么是 RAG？", "什么是向量数据库？"]:
    new_job = {"request_id": str(uuid.uuid4()), "query": q}
    processor.process(new_job)

print(f"\n总实际处理次数：{processor._call_count}（3 条唯一消息）")

=== 模拟消息重复投递 ===

投递第 1 次：
  ✅ [7b557029] 处理完成（第 1 次实际处理）

投递第 2 次：
  ⚠️  [7b557029] 重复消息，跳过（已处理过）

投递第 3 次：
  ⚠️  [7b557029] 重复消息，跳过（已处理过）

实际处理次数：1（应为 1，不是 3）

=== 新消息（不同 request_id）===
  ✅ [8d9d4594] 处理完成（第 2 次实际处理）
  ✅ [67954af5] 处理完成（第 3 次实际处理）

总实际处理次数：3（3 条唯一消息）


### 生产级幂等：教学版实现的三个缺陷

上面的 `IdempotentProcessor` 讲清了思路，但它**并不能真的防住并发重复**。落生产前必须补齐下面三点。

#### 缺陷 1：TOCTOU 竞态（check-then-act）

```mermaid
sequenceDiagram
    participant A as Consumer A
    participant R as Redis
    participant B as Consumer B
    A->>R: GET processed:rid
    R-->>A: None（未处理）
    B->>R: GET processed:rid
    R-->>B: None（未处理）
    A->>A: do_work() 执行
    B->>B: do_work() 执行（重复！）
```

两个 worker 同时消费同一条消息，都在检查阶段拿到 `None`，于是都进入处理。**检查与写入之间存在时间窗**，必须改成**先原子占位**：`SET lock:{rid} 1 NX EX 300`，只有一个消费者能抢到。

#### 缺陷 2："原子写入"跨了两个存储系统

`db.transaction()` 只能保证 DB 内部原子，`redis.setex` 是另一个系统的独立写入，两者之间没有分布式事务：

```mermaid
flowchart TB
    T["with db.transaction()"] --> D["db.save_result() 提交成功"]
    T --> R["redis.setex() 失败（网络抖动）"]
    D --> X1["下次重投 → 再执行一遍<br/>（重复，但结果一致，尚可接受）"]
    R -.->|反过来的顺序更危险| X2["标记写入成功，DB 回滚<br/>→ 结果不存在却被标记已处理<br/>（消息被静默丢弃）"]
```

> **结论**：真相只能有一份——放在 DB，用**唯一索引**兜底；Redis 只是前置加速器，它挂了或 key 过期了，正确性都不受影响。

#### 缺陷 3：缺"处理中"状态与租约（lease）

只有"处理完 / 没处理"两态，长任务容易卡死：消费者崩溃后标记残留 → 后续消息被**永久跳过**。生产上要三态 + 租约：

```mermaid
stateDiagram-v2
    [*] --> PENDING
    PENDING --> RUNNING: 抢到锁（lease_until = now + 300s）
    RUNNING --> DONE: 处理成功
    RUNNING --> PENDING: 崩溃 / 租约过期，可被重新领取
    DONE --> [*]
```

### 生产版实现：Redis 占位 + DB 唯一索引 + 租约

```python
class TransientError(Exception): ...   # 可重试：超时、限流、下游 5xx
class PermanentError(Exception): ...   # 不可重试：入参校验失败、权限不足


def process_job(job: dict):
    rid = job["request_id"]

    # ① 原子占位（NX：仅当键不存在才成功）+ 租约（EX 兜底，防永久锁死）
    if not redis.set(f"lock:{rid}", "1", nx=True, ex=300):
        existing = db.get_result(rid)
        if existing:
            return existing                     # 已完成：直接复用结果
        raise TransientError("in progress")     # 别人正在跑：交给重试策略等待

    try:
        result = do_work(job)                   # ② 业务逻辑（自身可重放）

        # ③ DB 唯一索引兜底，DB 是唯一真相
        row = db.execute("""
            INSERT INTO job_results(request_id, payload, status)
            VALUES (%s, %s, 'DONE')
            ON CONFLICT (request_id) DO NOTHING
            RETURNING payload
        """, rid, result)

        redis.setex(f"processed:{rid}", 86400, "1")   # ④ 纯加速，失败可忽略
        return row["payload"] if row else db.get_result(rid)

    except (TransientError, PermanentError):
        redis.delete(f"lock:{rid}")             # 释放租约；Permanent 不留"已处理"标记
        raise
```

**三层各司其职**

| 层 | 作用 | 挂了会怎样 |
|----|------|-----------|
| Redis `lock:{rid}`（`SET NX EX`） | 原子占位 + 租约，挡住并发重复 | 退化为"无并发保护"，DB 唯一索引仍兜底 |
| DB 唯一索引 `(request_id)` | **唯一真相**，最终正确性保证 | 系统不可用（这是正确性底线） |
| Redis `processed:{rid}`（`SETEX`） | 命中即短路，省一次 DB 查询 | 只是变慢，正确性不受影响 |

> 与教学版对比：**教学版用 Redis 当真相；生产版用 DB 当真相、Redis 只当加速器**。这是这段最重要的工程结论。

### `ON CONFLICT` / UPSERT 是什么

```sql
INSERT INTO job_results(request_id, payload, status) VALUES (?, ?, 'DONE')
ON CONFLICT (request_id) DO UPDATE
  SET payload = EXCLUDED.payload, status = 'DONE'
  RETURNING payload;
```

从 `ON CONFLICT` 开始的部分叫 **UPSERT**（PostgreSQL 9.5+）：先尝试 `INSERT`，撞上唯一约束时**不报错**，改为执行备用动作。

| 片段 | 含义 |
|------|------|
| `ON CONFLICT (request_id)` | **冲突目标**，必须对应一个真实存在的唯一索引；否则直接报错 |
| `DO NOTHING` | 静默跳过；但 `RETURNING` **拿不到行**，需再查一次 |
| `DO UPDATE` | 覆盖原行；插入 / 重复两种情况 `RETURNING` 结构一致 |
| `EXCLUDED` | **伪表**，代表"刚才试图插入的那一行" |

⚠️ **语义陷阱：`DO UPDATE` 是后写覆盖先写（last-write-wins）**

LLM 两次执行大概率结果不同，覆盖会破坏"第一次结果即最终结果"的契约。推荐：

```sql
-- ① 只认第一次（推荐，语义最干净）
ON CONFLICT (request_id) DO NOTHING

-- ② 保留 DO UPDATE 但加守卫：只填充空值 / 只更新未完成的行
ON CONFLICT (request_id) DO UPDATE
  SET payload = COALESCE(job_results.payload, EXCLUDED.payload),
      status  = 'DONE'
  WHERE job_results.status <> 'DONE'
```

| 数据库 | 等价写法 |
|--------|----------|
| PostgreSQL / SQLite | `ON CONFLICT (col) DO UPDATE SET ...` |
| MySQL | `ON DUPLICATE KEY UPDATE col = VALUES(col)` |
| SQL Server / Oracle | `MERGE INTO ... WHEN MATCHED THEN UPDATE` |

**并发细节**：两个事务同时插入同一 `request_id`，先到者插入、后到者在索引上阻塞等待；先到**提交** → 后到转入 `DO UPDATE`；先到**回滚** → 后到重试插入。所以 UPSERT 能正确吸收**并发重复**，远比"先 get 再 set"严格。

### 重试策略：Exponential Backoff + Jitter

```python
def retry_with_backoff(fn, max_retries=3, base_delay=1.0):
    for attempt in range(max_retries + 1):
        try:
            return fn()
        except TransientError as e:
            if attempt == max_retries:
                raise                    # 超过最大重试，走降级路径
            # Exponential backoff + jitter（防止惊群效应）
            delay = base_delay * (2 ** attempt) + random.uniform(0, 1)
            logger.warning(f"Retry {attempt+1}/{max_retries} in {delay:.2f}s: {e}")
            time.sleep(delay)
        except PermanentError:
            raise                        # 永久错误不重试（入参校验失败、权限不足）

# 延迟序列（base=1s）：立即 → 1~2s → 2~4s → 4~8s（超出则降级）
```

> **幂等与重试是一对，缺一不可**：
> - **重试**让失败能自愈 → 这是不丢消息的来源，也是 at-least-once 的成因
> - **幂等**让重复执行无害 → 吸收重试（以及崩溃重投）带来的重复
>
> 并且必须区分错误类型：`TransientError` 才重试；`PermanentError` 重试多少次都一样，直接上抛且**不要留下"已处理"标记**——否则用户修正输入后无法重跑。

## Part 8 · Backpressure 与流量控制

**什么是 Backpressure（背压）？** 当下游处理不过来时，主动向上游发出"慢一点"的信号，让整条链路按**最慢一环的速度**运行，而不是让数据无限堆积。名字借用了流体 / 电路的比喻：下游堵住，压力会**反过来（back）**传导到上游。

**为什么会产生？** 只要生产者比消费者快：

```mermaid
flowchart LR
    P["生产者<br/>100 msg/s"] --> Q[("队列 / Buffer")]
    Q --> C["消费者<br/>30 msg/s"]
```

生产者 100/s、消费者 30/s，队列每秒净增 70 条 → 内存耗尽（OOM）、消息延迟越来越大，最后整条链路雪崩。

**不处理会怎样**

- **内存爆掉**：无界队列把内存吃光
- **延迟失控**：消息在队列里排队，端到端延迟变得不可预测
- **雪崩**：下游越慢 → 积压越多 → 下游更崩溃（正反馈）

**应对策略**

| 策略 | 作用位置 | 实现 | 效果 |
|------|----------|------|------|
| 队列长度限制（有界队列） | 队列 | `maxsize=N`、`allkeys-lru` | 防止内存耗尽 |
| 阻塞式生产 | 生产者 | 队列满时 `put()` 阻塞 | 上游自动降到下游速度 |
| 背压信号（拒绝 / 降级） | 入口 | 队列长度 > 阈值时返回 429 / 503 | 快速失败优于慢挂 |
| 速率限制（限流） | 入口 | Token bucket / Redis rate limiter | 从源头削峰、保护下游 |
| 消费者并发控制 | 消费者 | Worker 数 × 每个 Worker 并发数 | 精细控制吞吐 |
| 批处理 / 合并 | 消费者 | 攒批一次处理多个请求 | 提高吞吐、降低成本 |
| Autoscaling | 消费者 | HPA 基于队列长度扩缩 Worker | 弹性应对流量波动 |

**Python 最小示例**

```python
import queue

q = queue.Queue(maxsize=100)   # 有界队列：最多缓存 100 个任务

q.put(job)          # 队列满时 put() 会阻塞，直到消费者取走 —— 这就是背压
# q.put_nowait(job) # 非阻塞：满则抛 queue.Full，需自己决定丢弃 / 拒绝
```

> ⚠️ 这个 Notebook 的 Pipeline 用的是**无限队列**（默认 `maxsize=0`），只起"缓冲"作用、并不真正背压；生产里要显式设 `maxsize` 才能防止积压失控。

> **一句话**：背压 = 把下游"扛不住"变成上游"慢下来" —— 用可控的降速 / 拒绝，换取系统不崩溃。

### 🧪 实操：有界队列 vs 无界队列（阻塞式背压）

下面用 `queue.Queue(maxsize=...)` 对比两种队列。生产者比消费者快得多：

```
生产者 ──put()──▶ [ 有界队列 maxsize=2 ] ──get()──▶ 消费者（慢）
                        ↑ 满时 put() 阻塞 = 背压
```

- **无界队列**（`maxsize=0`）：生产者不阻塞，任务全堆在内存里 → 队列峰值高
- **有界队列**（`maxsize=2`）：队列满时 `put()` 阻塞，生产者被迫降到消费者速度 → 峰值被压住

In [5]:
# ── 背压 Demo：无界队列 vs 有界队列（阻塞式背压）──────────────────────────────
import queue
import threading
import time

N_TASKS      = 6      # 总任务数
PRODUCE_GAP  = 0.05   # 生产者投递间隔（很快）
CONSUME_TIME = 0.20   # 消费者处理耗时（很慢）

def pipeline(maxsize: int):
    """跑一条 生产者 → 队列 → 消费者 流水线，返回 (总耗时, 队列峰值, 生产者阻塞总时长)"""
    q = queue.Queue(maxsize=maxsize)
    peak, blocked = 0, 0.0

    def producer():
        nonlocal peak, blocked
        for i in range(N_TASKS):
            t0 = time.perf_counter()
            q.put(f"task-{i}")               # ← 队列满时会阻塞，这就是背压
            blocked += time.perf_counter() - t0
            peak = max(peak, q.qsize())
            time.sleep(PRODUCE_GAP)
        q.put("__DONE__")                    # 结束哨兵

    def consumer():
        while True:
            item = q.get()
            if item == "__DONE__":
                break
            time.sleep(CONSUME_TIME)         # 模拟慢消费者
            q.task_done()

    t0 = time.perf_counter()
    p = threading.Thread(target=producer)
    c = threading.Thread(target=consumer)
    p.start(); c.start()
    p.join();  c.join()
    return round(time.perf_counter() - t0, 2), peak, round(blocked, 2)

print(f"生产 {N_TASKS} 个任务（每 {PRODUCE_GAP}s），消费者每个耗时 {CONSUME_TIME}s\n")

for label, ms in [("无界队列  maxsize=0", 0), ("有界队列  maxsize=2", 2)]:
    total, peak, blocked = pipeline(ms)
    print(f"{label} → 总耗时 {total}s | 队列峰值 {peak} 个 | 生产者阻塞 {blocked}s")

生产 6 个任务（每 0.05s），消费者每个耗时 0.2s

无界队列  maxsize=0 → 总耗时 1.21s | 队列峰值 4 个 | 生产者阻塞 0.0s
有界队列  maxsize=2 → 总耗时 1.22s | 队列峰值 2 个 | 生产者阻塞 0.34s


> **关键观察**：
> - 两种情况**总耗时都 ≈ 消费者处理时间**（瓶颈在消费者，堆多少任务都白搭）
> - 区别在于"压力去哪了"：无界队列 → 压力堆在**内存**（队列持续积压）；有界队列 → 压力回到**生产者**（被阻塞，峰值被压在 2）
> - 这就是背压的本质：**把"内存爆掉"换成"上游变慢"**
> - 若不想阻塞，可改用 `put_nowait()` + `except queue.Full` → 快速拒绝（对外返回 429 / 503）

## Part 9 · 可观测性（Metrics + Trace）

### 三大支柱

**可观测性（Observability）** = 系统出问题时，**仅凭它对外输出的数据**就能推断出内部发生了什么。业界把它归纳为**三大支柱（Three Pillars）**：

| 支柱 | 回答的问题 | 数据形态 | 典型工具 |
|------|------------|----------|----------|
| **Metrics**（指标） | "有没有出问题？" | 聚合数值（计数 / 延迟 / 错误率） | Prometheus + Grafana |
| **Traces**（链路追踪） | "问题出在哪一环？" | 单次请求的完整调用链 | OpenTelemetry / Jaeger |
| **Logs**（日志） | "到底发生了什么？" | 离散的结构化事件 | structlog / ELK |

> 一句话记忆：**Metrics 看趋势，Traces 看链路，Logs 看细节**。三者配合 —— 指标发现异常 → 链路定位到某个 Agent → 日志还原当时的具体细节。

---

#### Metrics（指标）
把系统状态聚合成**数值**（计数器 `Counter`、直方图 `Histogram`），最适合做监控大盘与告警。下面用 Prometheus 客户端示意：给每个 Agent 记录"请求次数"和"延迟分布"。

```python
from prometheus_client import Counter, Histogram

# 定义两个指标
requests_total = Counter(
    "agent_requests_total",  # 指标名
    "Total requests",        # 描述
    ["agent_name", "status"] # 标签（维度）
)
latency = Histogram(
    "agent_latency_seconds",
    "Request latency",
    ["agent_name"]
)

# 使用
with latency.labels("retriever").time():
    hits = do_retrieval(query)
requests_total.labels("retriever", "success").inc()
```
这段代码是用 Prometheus 客户端给「检索」这一步打监控埋点 —— 记录它被调用了多少次、耗时多少。
##### 1. 导入库
prometheus_client 是 Prometheus 官方的 Python 库，提供两种最常用的指标类型：
- Counter：计数器，只增不减（适合"请求次数 / 错误数"）
- Histogram：直方图，统计一段数值的分布（适合"延迟、响应大小"）
##### 2. 定义两个指标
```Python
requests_total = Counter(
    "agent_requests_total",        # ① 指标名
    "Total requests",              # ② 描述
    ["agent_name", "status"]       # ③ 标签（维度）
)
```
Counter(名字, 描述, 标签列表) 的三个参数：
1. 指标名：暴露给 Prometheus 用的名字。_total 是计数器的命名惯例。
2. 描述：一句人话说明，鼠标悬停/查询时能看到。
3. 标签 labelnames：给指标加"维度"。有了它，同一个指标能按不同维度分别统计 —— 这里就是"按哪个 Agent + 什么状态"。
```python
latency = Histogram(
    "agent_latency_seconds",   # 名字：_seconds 是"耗时"的命名惯例（基础单位是秒）
    "Request latency",
    ["agent_name"]
)
```
Histogram 会把每次观测到的耗时自动分桶（比如 ≤0.1s、≤0.5s、≤1s…），从而能算出 p50/p95/p99 这类分位数。

##### 3. 使用（打点）
```python
with latency.labels("retriever").time():
    hits = do_retrieval(query)
```
- `.labels("retriever")` → 取出一条` agent_name="retriever"` 的时间序列
- `.time()` → **上下文管理器**：进入 `with` 时计时，退出时把"耗时秒数"记进直方图
- 所以包在里面的 `do_retrieval(query)` 执行了多久，就被自动观测了
```python
requests_total.labels("retriever", "success").inc()
```
- 取 `agent_name="retriever", status="success"` 这条序列
- `.inc()` → **加 1**（`inc()` 默认加 1，也可 `inc(5)`）

#### Traces（链路追踪）
记录**一次请求在多个 Agent / 服务之间的完整流转**：每个环节是一个 `span`，能一眼看出是哪一环慢或失败。下面用 OpenTelemetry 给 `retrieve` 这一步打点。

```python
from opentelemetry import trace

tracer = trace.get_tracer("retriever")

with tracer.start_as_current_span("retrieve") as span:
    span.set_attribute("query", query)
    span.set_attribute("top_k", 20)
    hits = do_retrieval(query)
    span.set_attribute("hits_count", len(hits))
```

#### Logs（结构化日志）
把每一件发生的事记成**结构化文本**（`key=value`），用于精确还原现场、排查具体 bug。下面用 structlog 打印一条结构化的检索完成日志。

```python
import structlog
log = structlog.get_logger()

log.info("retrieval_done",
    request_id=job["request_id"],
    query=query,
    hits=len(hits),
    latency_ms=elapsed)
```

### 🧪 实操：可观测性（Metrics + Trace）

生产中用 Prometheus + OpenTelemetry，这里用纯 Python 模拟相同概念：
- **Metrics**：计数器、直方图（延迟分布）
- **Trace**：全链路 request_id，记录每个 Agent 的耗时与结果

In [ ]:
import statistics


# ── 简易 Metrics 收集器 ────────────────────────────────────────────────────
class Metrics:
    def __init__(self):
        self.counters: dict[str, int]       = defaultdict(int)
        self.histograms: dict[str, list]    = defaultdict(list)

    def inc(self, name: str, labels: dict = {}):
        key = name + str(sorted(labels.items()))
        self.counters[key] += 1

    def observe(self, name: str, value: float, labels: dict = {}):
        key = name + str(sorted(labels.items()))
        self.histograms[key].append(value)

    def summary(self):
        print("\n📊 Metrics Summary")
        print("─" * 50)
        for k, v in self.counters.items():
            print(f"  counter  {k} = {v}")
        for k, vals in self.histograms.items():
            p50 = statistics.median(vals)
            p95 = sorted(vals)[int(len(vals)*0.95)] if len(vals) > 1 else vals[0]
            print(f"  hist     {k}  p50={p50:.3f}s  p95={p95:.3f}s  n={len(vals)}")

metrics = Metrics()

# ── 带 Trace 和 Metrics 的 Pipeline ─────────────────────────────────────────
def traced_pipeline(query: str) -> dict:
    trace_id = str(uuid.uuid4())[:8]
    trace_log = []

    def span(name: str, fn, *args, **kwargs):
        t = time.perf_counter()
        result = fn(*args, **kwargs)
        elapsed = round(time.perf_counter() - t, 3)
        trace_log.append({"span": name, "latency_s": elapsed})
        metrics.observe("agent_latency_seconds", elapsed, {"agent": name})
        metrics.inc("agent_requests_total", {"agent": name, "status": "ok"})
        return result

    hits    = span("retriever", fake_retrieve, query, latency=0.3)
    topk    = span("reranker",  fake_rerank,   query, hits, latency=0.2)
    answer  = span("generator", fake_generate, query, topk, latency=0.5)
    verdict = span("verifier",  fake_verify,   answer, topk, latency=0.15)

    print(f"\n🔍 Trace [{trace_id}] for: {query[:30]}")
    for s in trace_log:
        bar = "█" * int(s["latency_s"] * 20)
        print(f"  {s['span']:<12} {bar:<20} {s['latency_s']}s")

    return {"answer": answer, "verdict": verdict, "trace_id": trace_id}

# ── 跑 5 次，生成 metrics ────────────────────────────────────────────────────
for q in ["RAG 是什么？", "幂等设计？", "可观测性？", "Blackboard？", "Pipeline？"]:
    traced_pipeline(q)

metrics.summary()


🔍 Trace [32dd9101] for: RAG 是什么？
  retriever    ██████               0.3s
  reranker     ████                 0.2s
  generator    ██████████           0.5s
  verifier     ███                  0.15s

🔍 Trace [bc402c63] for: 幂等设计？
  retriever    ██████               0.301s
  reranker     ████                 0.2s
  generator    ██████████           0.501s
  verifier     ███                  0.15s

🔍 Trace [88428d41] for: 可观测性？
  retriever    ██████               0.301s
  reranker     ████                 0.202s
  generator    ██████████           0.501s
  verifier     ███                  0.15s

🔍 Trace [ba54c293] for: Blackboard？
  retriever    ██████               0.3s
  reranker     ████                 0.2s
  generator    ██████████           0.5s
  verifier     ███                  0.15s

🔍 Trace [d877d384] for: Pipeline？
  retriever    ██████               0.301s
  reranker     ████                 0.201s
  generator    ██████████           0.5s
  verifier     ███                 

## Part 10 · 安全隔离与最小权限

**Agent 安全清单**

| 层次 | 措施 |
|------|------|
| **网络** | 各 Agent 只开放必要端口，使用 mTLS |
| **IAM** | 每个 Agent 独立 Service Account，最小权限 |
| **输入校验** | 所有入参用 Pydantic / jsonschema 校验 |
| **输出过滤** | LLM 输出过滤 Prompt Injection / PII |
| **速率限制** | 每个 Agent 入口限流（令牌桶） |
| **审计日志** | 记录所有 tool call 与外部 API 调用 |

> **关键原则**：绝不直接执行 LLM 生成的命令，必须经过参数白名单校验

In [12]:
# ── 输入校验：白名单优先 ─────────────────────────────────────────────────────
ALLOWED_TASKS   = {'summarize', 'extract_risks', 'propose_measures'}
ALLOWED_DOC_IDS = {f'policy-{y}' for y in ['2023', '2024', '2025']}

@dataclass
class TaskRequest:
    doc_id: str
    task:   str
    metadata: dict = field(default_factory=dict)

    def validate(self):
        errors = []
        if not isinstance(self.doc_id, str) or len(self.doc_id) > 64:
            errors.append(f'doc_id invalid: {self.doc_id!r}')
        if self.doc_id not in ALLOWED_DOC_IDS:
            errors.append(f'doc_id not in whitelist: {self.doc_id!r}')
        if self.task not in ALLOWED_TASKS:
            errors.append(f'task not in whitelist: {self.task!r}')
        if errors:
            raise ValueError('Validation failed: ' + '; '.join(errors))
        return self


# ── 权限矩阵：最小权限原则 ───────────────────────────────────────────────────
PERMISSIONS = {
    'retriever':      {'read_docs': True,  'write_db': False, 'call_llm': False, 'send_alert': False},
    'summarizer':     {'read_docs': True,  'write_db': False, 'call_llm': True,  'send_alert': False},
    'risk_extractor': {'read_docs': True,  'write_db': False, 'call_llm': True,  'send_alert': False},
    'verifier':       {'read_docs': True,  'write_db': False, 'call_llm': True,  'send_alert': False},
    'supervisor':     {'read_docs': True,  'write_db': True,  'call_llm': False, 'send_alert': True},
}

def check_permission(agent, action):
    return PERMISSIONS.get(agent, {}).get(action, False)


print('=== Input Validation Tests ===\n')
test_cases = [
    TaskRequest('policy-2025',  'summarize'),
    TaskRequest('policy-2025',  'DROP TABLE users; --'),
    TaskRequest('secret-admin', 'summarize'),
    TaskRequest('policy-2025',  'delete_all'),
    TaskRequest('policy-2024',  'extract_risks'),
]
for req in test_cases:
    try:
        req.validate()
        print(f'  ALLOWED: doc={req.doc_id!r}  task={req.task!r}')
    except ValueError as e:
        print(f'  BLOCKED: {e}')

print('\n=== Permission Matrix Tests ===\n')
tests = [
    ('retriever',  'read_docs'),
    ('retriever',  'write_db'),
    ('summarizer', 'call_llm'),
    ('summarizer', 'send_alert'),
    ('supervisor', 'write_db'),
    ('supervisor', 'send_alert'),
]
for agent, action in tests:
    ok = check_permission(agent, action)
    status = 'ALLOWED' if ok else 'DENIED '
    print(f'  {status}: {agent:<18} -> {action}')

print('\nSecurity Summary:')
print('  1. All external inputs validated against whitelist')
print('  2. Each Agent has only minimum permissions for its role')
print('  3. Rejection events logged (printed here; production: write to audit DB)')
print('  4. SQL injection blocked at validation layer, never reaches Workers')

=== Input Validation Tests ===

  ALLOWED: doc='policy-2025'  task='summarize'
  BLOCKED: Validation failed: task not in whitelist: 'DROP TABLE users; --'
  BLOCKED: Validation failed: doc_id not in whitelist: 'secret-admin'
  BLOCKED: Validation failed: task not in whitelist: 'delete_all'
  ALLOWED: doc='policy-2024'  task='extract_risks'

=== Permission Matrix Tests ===

  ALLOWED: retriever          -> read_docs
  DENIED : retriever          -> write_db
  ALLOWED: summarizer         -> call_llm
  DENIED : summarizer         -> send_alert
  ALLOWED: supervisor         -> write_db
  ALLOWED: supervisor         -> send_alert

Security Summary:
  1. All external inputs validated against whitelist
  2. Each Agent has only minimum permissions for its role
  3. Rejection events logged (printed here; production: write to audit DB)
  4. SQL injection blocked at validation layer, never reaches Workers


## Part 11 · 成本优化策略

### 分层模型路由
```python
def select_model(task_type: str) -> str:
    routing = {
        "retrieval_score":  "haiku-4-5",   # 便宜
        "rerank":           "haiku-4-5",
        "generation_simple":"sonnet-4-6",  # 主力
        "generation_complex":"opus-4-6",   # 贵，按需
        "verification":     "haiku-4-5",
    }
    return routing.get(task_type, "sonnet-4-6")
```

### 缓存策略
```python
import hashlib

def cached_retrieve(query: str):
    key = f"cache:retrieve:{hashlib.md5(query.encode()).hexdigest()}"
    cached = redis.get(key)
    if cached:
        return json.loads(cached)
    result = do_retrieval(query)
    redis.setex(key, 3600, json.dumps(result))
    return result
```

### 批处理
- 聚合多个请求一次调用 LLM
- 使用 Embedding batch API

## Part 12 · 优雅降级（Graceful Degradation）

**场景**：Reranker 服务不可用时，系统不应整体崩溃，而应**降级**继续提供服务。

```
正常路径：  retrieve → rerank → generate  ✅
降级路径1： retrieve → (skip rerank) → generate  ⚠️ 质量略降
降级路径2： (retrieve timeout) → return cached snippets  ⚠️ 更大降级
兜底路径：  return static fallback message  🆘
```

**降级层次**：完整 Pipeline → 跳过 Reranker → 仅返回检索片段 → 缓存结果 → 静态兜底

> 用 `try/except` 逐级捕获失败，每一层都退到一个"还能用"的降级方案 —— 详见本 Part 的实操 `ResilientOrchestrator`。

In [9]:
class RerankerUnavailable(Exception): pass
class RetrieverTimeout(Exception): pass

class ResilientOrchestrator:
    def __init__(self, reranker_up: bool = True, retriever_up: bool = True):
        self.reranker_up  = reranker_up
        self.retriever_up = retriever_up
        self._cache: dict[str, list] = {}

    def handle(self, query: str) -> dict:
        degraded_reason = None

        # ── 检索（有缓存则用缓存）────────────────────────────────────────────
        try:
            if not self.retriever_up:
                raise RetrieverTimeout("retriever unavailable")
            hits = fake_retrieve(query, latency=0.1)
            self._cache[query] = hits          # 写缓存
        except RetrieverTimeout:
            hits = self._cache.get(query, [])
            if hits:
                degraded_reason = "retriever_timeout_used_cache"
            else:
                return {
                    "answer": "暂时无法检索，请稍后重试。",
                    "degraded": True,
                    "reason": "retriever_down_no_cache",
                }

        # ── 重排（降级：跳过，直接用粗检结果）───────────────────────────────
        try:
            if not self.reranker_up:
                raise RerankerUnavailable("reranker is down")
            topk = fake_rerank(query, hits, top_k=3, latency=0.1)
        except RerankerUnavailable:
            topk = hits[:3]                    # 降级：直接用 top-3 粗检结果
            degraded_reason = degraded_reason or "reranker_down_using_raw_hits"

        # ── 生成 ─────────────────────────────────────────────────────────────
        answer = fake_generate(query, topk, latency=0.3)
        return {
            "answer": answer[:80],
            "degraded": degraded_reason is not None,
            "reason": degraded_reason,
        }


# ── 测试三种场景 ──────────────────────────────────────────────────────────────
scenarios = [
    ("✅ 正常路径",          dict(reranker_up=True,  retriever_up=True)),
    ("⚠️  Reranker 故障",   dict(reranker_up=False, retriever_up=True)),
    ("🆘 Retriever 故障",   dict(reranker_up=True,  retriever_up=False)),
]

query = "什么是优雅降级？"
for label, flags in scenarios:
    orc = ResilientOrchestrator(**flags)
    r   = orc.handle(query)
    status = "降级" if r["degraded"] else "正常"
    print(f"{label}")
    print(f"  状态：{status}  原因：{r.get('reason', 'none')}")
    print(f"  答案：{r['answer'][:60]}...")
    print()

✅ 正常路径
  状态：正常  原因：None
  答案：[答案] 关于「什么是优雅降级？」：消息队列解耦生产者和消费者，支持异步处理。 Blackboard 模式通过共享工作区...

⚠️  Reranker 故障
  状态：降级  原因：reranker_down_using_raw_hits
  答案：[答案] 关于「什么是优雅降级？」：消息队列解耦生产者和消费者，支持异步处理。 向量数据库支持语义相似度搜索。......

🆘 Retriever 故障
  状态：降级  原因：retriever_down_no_cache
  答案：暂时无法检索，请稍后重试。...



> **关键观察**：
> - Reranker 故障时，系统仍能返回答案（质量略降）
> - Retriever 故障时，优先用缓存；无缓存才返回兜底消息
> - 每个降级路径都有明确的 `reason`，便于监控告警

## Part 13 · 进阶：三个完整实现示例

> 用三种真实技术栈，把前面学到的 **Pipeline / Hub-and-Spoke / Blackboard** 各落地一遍。

### 示例概览

| 示例 | 模式 | 技术栈 | 难度 |
|------|------|--------|------|
| **示例 1** | Pipeline | Redis + Python Workers | ⭐ 入门 |
| **示例 2** | Hub-and-Spoke | FastAPI + HTTP | ⭐⭐ 进阶 |
| **示例 3** | Blackboard | DB + Pub-Sub | ⭐⭐⭐ 高级 |

> 每个示例均可在本地独立运行，建议按顺序动手实现。

---

### 示例 1 · Redis Pipeline

#### 架构

```
用户请求
   │
   ▼
Orchestrator ──push──▶ jobs:retriever
                              │
                        RetrieverWorker
                              │ push
                              ▼
                        jobs:reranker
                              │
                        RerankerWorker
                              │ push
                              ▼
                        jobs:generator
                              │
                        GeneratorWorker ──write──▶ results:{job_id}
                                                        │
                                                        ▼
                                                  API 轮询取结果
```

#### 核心代码

**Orchestrator — 提交任务**
```python
# orchestrator.py
import redis, json, uuid

r = redis.Redis()

def submit_query(query: str) -> str:
    job_id = str(uuid.uuid4())
    job = {"request_id": job_id, "query": query, "stage": "retriever"}
    r.lpush("jobs:retriever", json.dumps(job))
    return job_id
```

**Retriever Worker — 消费 & 转发**
```python
# retriever_worker.py
while True:
    _, data = r.brpop("jobs:retriever")   # blocking pop
    job = json.loads(data)

    # 幂等检查
    if r.get(f"done:retriever:{job['request_id']}"):
        continue

    job["hits"] = do_retrieval(job["query"], top_k=20)
    job["stage"] = "reranker"
    r.lpush("jobs:reranker", json.dumps(job))
    r.setex(f"done:retriever:{job['request_id']}", 3600, 1)
```

**Reranker Worker**
```python
# reranker_worker.py
while True:
    _, data = r.brpop("jobs:reranker")
    job = json.loads(data)
    job["topk"] = cross_encoder_rerank(job["query"], job["hits"], top_k=5)
    job["stage"] = "generator"
    r.lpush("jobs:generator", json.dumps(job))
```

**Generator Worker**
```python
# generator_worker.py
while True:
    _, data = r.brpop("jobs:generator")
    job = json.loads(data)
    context = "\n".join([h["text"] for h in job["topk"]])
    answer = llm_generate(question=job["query"], context=context)
    r.setex(f"result:{job['request_id']}", 3600, json.dumps({"answer": answer}))
```

> **运行方式**：分别在 3 个终端启动三个 Worker，再调用 `submit_query()`。
```bash
# 终端 1
python retriever_worker.py

# 终端 2
python reranker_worker.py

# 终端 3
python generator_worker.py

# 终端 4（投递任务 + 取结果）
python -c "from orchestrator import submit_query; print(submit_query('什么是 RAG？'))"
```

---

### 示例 2 · HTTP Orchestrator（Hub-and-Spoke）

#### 架构

```
用户
 │ POST /query
 ▼
┌─────────────────────────────────┐
│         Orchestrator            │
│  1. POST /retrieve → hits       │
│  2. POST /rerank   → topk       │
│  3. POST /generate → answer     │
│  4. POST /verify   → verdict    │
└─────────────────────────────────┘
     │         │         │         │
 :8001      :8002     :8003     :8004
Retriever  Reranker Generator Verifier
```

#### 核心代码

```python
# orchestrator_http.py
import httpx
from fastapi import FastAPI

app = FastAPI()
SERVICES = {
    "retriever": "http://localhost:8001",
    "reranker":  "http://localhost:8002",
    "generator": "http://localhost:8003",
    "verifier":  "http://localhost:8004",
}

@app.post("/query")
async def handle_query(payload: dict):
    q = payload["query"]
    async with httpx.AsyncClient(timeout=30) as client:
        hits    = (await client.post(f"{SERVICES['retriever']}/retrieve",
                                      json={"query": q})).json()["hits"]
        topk    = (await client.post(f"{SERVICES['reranker']}/rerank",
                                      json={"query": q, "candidates": hits})).json()["topk"]
        answer  = (await client.post(f"{SERVICES['generator']}/generate",
                                      json={"question": q, "context": topk})).json()["answer"]
        verdict = (await client.post(f"{SERVICES['verifier']}/check",
                                      json={"answer": answer, "context": topk})).json()
    return {"answer": answer, "verdict": verdict}
```

> 每个微服务独立用 FastAPI 实现，用 Docker Compose 一键启动。

---

### 示例 3 · Blackboard（共享工作区）

#### 核心思路
共享数据库作为"黑板"，Agent 通过事件感知变化。

#### 核心代码

```python
# blackboard.py — 简化示意
import redis

r = redis.Redis()

class Blackboard:
    def write(self, task_id: str, field: str, value):
        """写入字段并发布变更事件"""
        r.hset(f"task:{task_id}", field, json.dumps(value))
        r.publish(f"task:{task_id}:events", field)

    def read(self, task_id: str, field: str):
        raw = r.hget(f"task:{task_id}", field)
        return json.loads(raw) if raw else None

    def subscribe(self, task_id: str):
        """订阅该任务的所有变更"""
        pubsub = r.pubsub()
        pubsub.subscribe(f"task:{task_id}:events")
        return pubsub
```

#### 流程
Ingest Agent 写 `chunks` → Retriever 订阅 `new_query` 事件并写 `hits` → Reranker 订阅 `hits_ready` 并写 `topk` → Generator 读取 `topk` 生成并写 `answer`

---

# Part 14 · 分层架构：Worker / Orchestrator / Supervisor

## 📖 概念

### 14.1 从"一个 Agent 干所有事"到"分层协作"

同一个任务（"为企业政策文档生成三页报告：要点总结 / 风险清单 / 建议措施"），两种做法：

**方案 A：单体 Agent**

```mermaid
flowchart TB
    User[用户] --> One[one_agent document]
    One --> Report[报告]
    One --> Tools["一个 LLM + 所有工具<br/>检索 / 摘要 / 风险提取<br/>建议生成 / 事实校验 / 格式化"]
```

问题：
- Context 窗口装不下长文档 + 全部工具调用历史
- 一步出错，整体失败，无法定位哪步错了
- 无法优化单个能力（换 Retriever 要改整个 Agent）
- 生产后：出了问题谁负责？质量如何保证？

**方案 B：分层多 Agent + Supervision**

```mermaid
flowchart TB
    Supervisor["Supervisor<br/>质量门 / HITL / 合规审计"]
    Orchestrator["Orchestrator<br/>计划 / 分派 / 汇总"]
    Retriever["Retriever Agent"]
    Summarizer["Summarizer"]
    RiskExtract["RiskExtract Agent"]
    Verifier["Verifier<br/>NLI 事实校验"]

    Supervisor --> Orchestrator
    Orchestrator -->|"并行"| Retriever
    Orchestrator -->|"并行"| Summarizer
    Orchestrator -->|"并行"| RiskExtract
    Retriever --> Verifier
    Summarizer --> Verifier
    RiskExtract --> Verifier
```

### 14.2 分层带来的四大收益

| 收益 | 说明 |
|------|------|
| **职责清晰** | 每个组件只做一件事，易测试、易替换、易优化 |
| **独立扩容** | Retriever 高并发 → 多副本；Generator 成本高 → 精细限流 |
| **故障隔离** | Reranker 挂了 → 降级，不影响其他；系统有弹性 |
| **可观测 & 可治理** | 每步有 trace；Verifier 自动校验；Supervisor 记审计 + HITL |

### 14.3 三层全貌

#### 顶层：Supervisor（策略 & 治理）
- 设置质量阈值、监听所有 trace、触发 Quality Gate
- 创建 HITL ticket、记录审计日志、发出告警
- **不做**：具体任务

#### 中层：Orchestrator（计划 & 协调）
- 解析意图 → 生成执行计划 → 分派子任务 → 收集结果 → 汇总 → 调用 Verifier
- **不做**：检索 / 生成 / 校验

#### 底层：Worker Agents（单一技能）
- Retriever / Summarizer / RiskExtractor / MeasureProposer / FactChecker ...
- **不做**：计划 / 汇总 / 监督

> **黄金法则**：每层只做自己该做的事，通过**接口（消息 / API）**交互，不直接调用内部实现。

### 14.4 Supervisor 职责示意

```python
class Supervisor:
    min_pass_rate    = 0.80    # Verifier pass_rate 下限
    max_latency_p95  = 5000    # ms，超出触发告警
    max_cost_per_job = 0.50    # USD，超出触发审批

    def on_job_complete(self, job_id, result):
        verdict = result["verdict"]
        if verdict["pass_rate"] < self.min_pass_rate:
            return self.create_hitl_ticket(job_id, reason="low_quality")
        if result["cost_usd"] > self.max_cost_per_job:
            return self.create_hitl_ticket(job_id, reason="cost_exceeded")
        self.write_audit_log(job_id, result, status="auto_approved")
        return {"action": "proceed"}
```

Supervisor 只是**观察者 + 决策者**，不参与具体业务逻辑。

### 14.5 Orchestrator 的五步法

```mermaid
flowchart LR
    Start["handle_request"] --> Plan["1. plan<br/>分解子任务"]
    Plan --> Dispatch["2. dispatch_all<br/>asyncio.gather"]
    Dispatch --> Aggregate["3. aggregate<br/>合并结果"]
    Aggregate --> Verify["4. verify<br/>Verifier 校验"]
    Verify --> Report["5. report_to_supervisor"]
```

计划可以是**静态**（预定义子任务）或**动态**（LLM 生成，见 Part 2）。

### 14.6 Worker 设计原则

**单一职责 + 无状态 + 幂等**

```python
class WorkerAgent:
    name: str
    input_schema: dict
    output_schema: dict

    def run(self, job: dict) -> dict:
        # 1. 校验输入 schema   2. 执行单一职责
        # 3. 返回结构化输出     4. 不保存任务状态到 self
        ...
```

| 职责 | 实现 A | 实现 B | 切换方式 |
|------|--------|--------|----------|
| 检索 | BM25 | FAISS 向量 | 改 `retriever=` 配置 |
| 重排 | BM25 Score | CrossEncoder | 改 `reranker=` 配置 |
| 生成 | GPT-4o | Claude Sonnet | 改 `generator=` 配置 |
| 校验 | NLI Model | LLM-as-Judge | 改 `verifier=` 配置 |

### 14.7 权衡

**收益**：每层可独立测试 / 独立扩容 / Worker 可替换 / Trace 精确到每一步 / Supervisor 与业务解耦

**代价**：代码量更多（需定义接口）/ 调用链更长（增加网络延迟）/ 初期设计更复杂 / 调试需跨层追踪

**不适合**：一次性脚本、极简单的单步任务、延迟要求 < 100ms 的实时系统

> **建议**：先做单体 MVP 验证需求，再演进为分层架构。

### 14.8 小练习

场景："智能简历筛选系统"，请画出分层架构：
1. **识别 Worker Agents**：ResumeParser、SkillMatcher、ExperienceScorer、CultureFitEvaluator
2. **定义 Orchestrator 计划**：Parse → 并行(SkillMatch, ExperienceScore) → Aggregate → VerifyTop3
3. **设计 Supervisor 质量门**：匹配率 < 60% 或有争议候选 → HITL
4. **HITL 触发点**：最终推荐名单的人工确认，尤其是 Reject 决策

---

## 💻 实操

下面的 Demo 使用**政策文档语料**（与 Part 0 的 RAG 语料互不影响），并额外提供 `TraceStore`：按 `job_id` 聚合全链路记录，可按 job 打印延迟条。

In [6]:
# ── 政策文档语料（与 Part 0 的 RAG 语料互不影响）────────────────────────────
POLICY_CORPUS = [
    "公司差旅政策规定，国内出差需提前 3 天申请审批，国际出差需提前 10 天。",
    "所有采购金额超过 5 万元须经部门经理和 CFO 双签审批。",
    "数据安全策略要求敏感数据必须加密存储，禁止存储于个人设备。",
    "员工绩效评估每半年进行一次，需由直线经理和 HR 共同完成。",
    "供应商合同须经法务部门审核，合同期超过 1 年需 CEO 签字。",
    "信息安全事故须在发现后 24 小时内上报 CISO，并留存完整日志。",
    "远程办公政策允许每周最多 3 天居家，需提前与团队确认。",
    "知识产权归公司所有，员工在职期间的相关创作须签署归属协议。",
    "反腐败政策禁止收受任何形式的礼品，超过 100 元的礼品须上交。",
    "紧急事件响应预案要求关键系统 RTO <= 4 小时，RPO <= 1 小时。",
]

def policy_retrieve(query, top_k=5, latency=0.2):
    time.sleep(latency)
    scored = [(doc, random.uniform(0.5, 1.0)) for doc in POLICY_CORPUS]
    scored.sort(key=lambda x: -x[1])
    return [{"text": doc, "score": round(s, 3)} for doc, s in scored[:top_k]]

def policy_summarize(chunks, latency=0.3):
    time.sleep(latency)
    snippet = chunks[0]["text"][:40] if chunks else "(empty)"
    return f"[Summary] Policy highlights: {snippet}... ({len(chunks)} items)"

def policy_extract_risks(chunks, latency=0.25):
    time.sleep(latency)
    keywords = ["禁止", "须", "必须", "规定", "要求"]
    risks = []
    for c in chunks[:3]:
        for kw in keywords:
            if kw in c["text"]:
                risks.append(f"Risk: violates [{c['text'][:30]}...]")
                break
    return risks or ["Risk: no obvious violations found"]

def policy_propose_measures(risks, latency=0.2):
    time.sleep(latency)
    return [f"Measure: for [{r[:25]}...], strengthen approval process" for r in risks[:3]]

def policy_nli_check(context, claim):
    time.sleep(0.05)
    return random.random() > 0.2


# ── TraceStore：按 job_id 聚合全链路记录 ────────────────────────────────────
class TraceStore:
    def __init__(self):
        self._traces = []

    def record(self, job_id, agent, step, inp, out, latency_s, extra=None):
        entry = {
            "job_id":     job_id,
            "agent":      agent,
            "step":       step,
            "input":      str(inp)[:100],
            "output":     str(out)[:100],
            "latency_ms": round(latency_s * 1000),
            "timestamp":  round(time.time(), 3),
        }
        if extra:
            entry.update(extra)
        self._traces.append(entry)

    def get_job_trace(self, job_id):
        return [t for t in self._traces if t["job_id"] == job_id]

    def print_job_trace(self, job_id):
        entries = self.get_job_trace(job_id)
        print(f"\nTrace [{job_id[:8]}]  ({len(entries)} steps)")
        print("-" * 56)
        for e in entries:
            bar = "#" * min(int(e["latency_ms"] / 80), 18)
            print(f"  step {e['step']:2d} | {e['agent']:<18} | {bar:<18} {e['latency_ms']}ms")
        total = sum(e["latency_ms"] for e in entries)
        print("-" * 56)
        print(f"  Total: {len(entries)} entries, sum latency: {total}ms")


TRACE = TraceStore()
print("OK: policy tools + TraceStore ready")

OK: policy tools + TraceStore ready


In [7]:
# ── Worker Agents（底层：单一技能）──────────────────────────────────────────
class RetrieverWorker:
    name = "retriever"
    def run(self, doc_id):
        chunks = policy_retrieve(f"policy {doc_id}", top_k=6, latency=0.2)
        return {"chunks": chunks, "count": len(chunks)}

class SummarizerWorker:
    name = "summarizer"
    def run(self, chunks):
        return {"summary": policy_summarize(chunks, latency=0.3)}

class RiskExtractorWorker:
    name = "risk_extractor"
    def run(self, chunks):
        return {"risks": policy_extract_risks(chunks, latency=0.25)}

class MeasureProposerWorker:
    name = "measure_proposer"
    def run(self, risks):
        return {"measures": policy_propose_measures(risks, latency=0.2)}


# ── Supervisor（顶层：策略 & 治理）──────────────────────────────────────────
class Supervisor:
    def __init__(self, min_pass_rate=0.80):
        self.min_pass_rate = min_pass_rate
        self.jobs = {}

    def start_job(self, job_id, doc_id):
        self.jobs[job_id] = {'doc_id': doc_id, 'status': 'running', 'started': time.time()}
        print(f"  [Supervisor] START job {job_id[:8]}")

    def quality_gate(self, job_id, pass_rate):
        if pass_rate >= self.min_pass_rate:
            return {'action': 'proceed',
                    'reason': f'pass_rate={pass_rate:.2f} >= {self.min_pass_rate}'}
        return {'action': 'human_review',
                'reason': f'pass_rate={pass_rate:.2f} < {self.min_pass_rate}'}

    def close_job(self, job_id, status):
        if job_id in self.jobs:
            self.jobs[job_id]['status'] = status
        print(f"  [Supervisor] CLOSE job {job_id[:8]} -> {status}")

    def create_hitl_ticket(self, job_id, reason):
        tid = f"TICKET-{job_id[:6].upper()}"
        print(f"  [Supervisor] HITL ticket: {tid}  ({reason})")
        return tid


# ── Orchestrator（中层：计划 & 协调）────────────────────────────────────────
class SimpleOrchestrator:
    def __init__(self):
        self.retriever  = RetrieverWorker()
        self.summarizer = SummarizerWorker()
        self.risk_ext   = RiskExtractorWorker()
        self.measure    = MeasureProposerWorker()
        self.supervisor = Supervisor()

    def handle(self, doc_id):
        job_id = str(uuid.uuid4())
        self.supervisor.start_job(job_id, doc_id)
        step = [0]

        def run(worker, *args):
            step[0] += 1
            t = time.perf_counter()
            res = worker.run(*args)
            lat = time.perf_counter() - t
            TRACE.record(job_id, worker.name, step[0], str(args)[:40], res, lat)
            print(f"  [{worker.name:<18}] OK {round(lat, 3)}s")
            return res

        ret   = run(self.retriever, doc_id)
        summ  = run(self.summarizer, ret['chunks'])
        risks = run(self.risk_ext,   ret['chunks'])
        meas  = run(self.measure,    risks['risks'])

        self.supervisor.close_job(job_id, 'passed')
        TRACE.print_job_trace(job_id)
        return {'job_id': job_id, **summ, **risks, **meas}


orc = SimpleOrchestrator()
t0 = time.perf_counter()
report = orc.handle("policy-2025")
print(f"\nTotal: {round(time.perf_counter() - t0, 3)}s")
print(f"Summary: {report['summary'][:70]}")

  [Supervisor] START job 289a55fa
  [retriever         ] OK 0.205s
  [summarizer        ] OK 0.305s
  [risk_extractor    ] OK 0.255s
  [measure_proposer  ] OK 0.201s
  [Supervisor] CLOSE job 289a55fa -> passed

Trace [289a55fa]  (4 steps)
--------------------------------------------------------
  step  1 | retriever          | ##                 205ms
  step  2 | summarizer         | ###                305ms
  step  3 | risk_extractor     | ###                255ms
  step  4 | measure_proposer   | ##                 201ms
--------------------------------------------------------
  Total: 4 entries, sum latency: 966ms

Total: 0.968s
Summary: [Summary] Policy highlights: 远程办公政策允许每周最多 3 天居家，需提前与团队确认。... (6 items)


# Part 15 · Agent Supervision：Trace、质量门、HITL、审计

## 📖 概念

### 15.1 为什么 Supervision 不可缺

没有 Supervision 的多 Agent 系统，以下问题都回答不了：

- 这个答案是怎么得出的？哪一步最耗时、哪一步出了错？
- 输出质量达标了吗？谁来保证？
- 用了什么模型、多少 token、多少钱？
- 万一合规审计，有日志可查吗？

| 问题类型 | 解决手段 |
|----------|----------|
| 质量不可知 | Verifier + Quality Gate |
| 故障无法定位 | 全链路 Trace |
| 高风险决策 | Human-in-the-Loop |
| 合规 / 法务审计 | Immutable Audit Log |
| 成本 / 安全失控 | Policy + Budget Gate |

### 15.2 Trace 设计：捕获什么

每个 Agent 每次调用产生一条 Trace Entry：

```json
{
  "trace_id":   "t-uuid",
  "job_id":     "j-uuid",
  "agent":      "summarizer",
  "step":       3,
  "input":  { "chunks": ["..."], "top_k": 5 },
  "output": { "summary": "..." },
  "start_time": "2025-01-15T10:30:00.000Z",
  "end_time":   "2025-01-15T10:30:01.200Z",
  "metrics": {
    "latency_ms": 1200, "tokens_in": 850, "tokens_out": 320,
    "model": "claude-haiku-4-5", "cost_usd": 0.0012
  },
  "parent_trace_id": "t-uuid-orchestrator"
}
```

**关键字段**：`job_id`（全链路聚合）、`parent_trace_id`（层级关系）、`metrics`（性能与成本）

```mermaid
flowchart TB
    Job["job_id: j-001"]
    Orch["orchestrator step 0 · 2.5s"]
    Ret["retriever step 1 · 0.3s"]
    Sum["summarizer step 2 · 1.2s"]
    Ver["verifier step 3 · 0.6s"]
    Gate["quality_gate pass_rate=0.67 → human_review"]
    Job --> Orch --> Ret --> Sum --> Ver --> Gate
```

生产工具：**Langfuse / LangSmith**（LLM 专用 trace）、**OpenTelemetry + Jaeger**（分布式追踪）、自研 TraceStore（教学 / 轻量场景）。

### 15.3 Quality Gate：量化质量

```python
@dataclass
class QualityConfig:
    min_pass_rate:    float = 0.80   # Verifier NLI pass rate
    min_faithfulness: float = 0.75   # RAGAS faithfulness
    max_latency_p95:  int   = 5000   # ms
    max_cost_per_job: float = 0.50   # USD

def quality_gate(result: dict, cfg: QualityConfig) -> dict:
    checks = {
        "verifier_pass": result["pass_rate"] >= cfg.min_pass_rate,
        "faithfulness":  result.get("faithfulness", 1.0) >= cfg.min_faithfulness,
        "latency_ok":    result["latency_ms"] <= cfg.max_latency_p95,
        "cost_ok":       result.get("cost_usd", 0) <= cfg.max_cost_per_job,
    }
    n_failed = sum(1 for v in checks.values() if not v)
    if n_failed == 0: return {"action": "proceed",         "level": "INFO"}
    if n_failed == 1: return {"action": "flag_for_review", "level": "WARN"}
    return               {"action": "human_review",        "level": "BLOCK"}
```

### 15.4 HITL 分级

| 级别 | 触发条件 | 处理方式 | 等待人工？ |
|------|----------|----------|-----------|
| **INFO** | 全部指标正常 | 自动通过，记录 trace | ❌ |
| **WARN** | 1 项指标低于阈值 | 标注存疑，后台异步复核 | ❌（非阻塞） |
| **REVIEW** | 2+ 项低或高风险 | 创建 ticket，等待审核 | ✅（阻塞） |
| **BLOCK** | 安全规则 / PII / 法务 | 立即停止，通知安全团队 | ✅（强制） |

```mermaid
flowchart TB
    T["Supervisor 创建 ticket<br/>含 trace / QA 报告 / Verifier 细节"]
    UI["审核界面 Slack / 内部工具 / 邮件"]
    Dec["审核人决策：批准 / 拒绝 / 要求修改"]
    Act["Orchestrator 继续 / 回滚 / 触发重新生成"]
    T --> UI --> Dec --> Act
```

**挂起与恢复**

```python
# 挂起 job：保存完整上下文（供恢复使用）
def suspend_job(job_id, ticket_id):
    db.update(job_id, status="suspended", ticket_id=ticket_id)
    ctx_store.save(job_id, full_context)

# 人工决策后恢复
def resume_job(job_id, decision):
    ctx = ctx_store.load(job_id)
    if decision == "approve":  continue_from_checkpoint(ctx)
    elif decision == "revise": re_run_with_feedback(ctx, feedback=decision["notes"])
    else:                      mark_failed(job_id)
```

**给审核人的 ticket 必须信息充分**

```json
{
  "ticket_id": "TICKET-ABC123",
  "summary":   "3 页政策报告草稿",
  "quality": {
    "pass_rate": 0.67,
    "failed_claims": ["风险条款 2 无文档支撑", "建议措施 1 与策略矛盾"]
  },
  "trace_url": "https://langfuse/job/abc123",
  "suggested_action": "REVISE",
  "auto_deadline": "2025-01-16T10:00:00Z"
}
```

### 15.5 Audit Log：合规与回溯

```json
{
  "event_id":   "audit-uuid",
  "job_id":     "j-uuid",
  "event_type": "human_review_decision",
  "timestamp":  "2025-01-15T11:30:00Z",
  "actor":      "alice@company.com",
  "decision":   "approve",
  "reason":     "Content verified by domain expert",
  "overrides": { "original_pass_rate": 0.67, "approved_despite_low_quality": true },
  "ip_address": "10.0.0.1",
  "session_id": "sess-uuid"
}
```

**存储要求**：Append-only（不可修改）、加密、访问控制、保留期通常 5–7 年。
**工具**：AWS CloudTrail、Elasticsearch + ILM、PostgreSQL audit trigger。

---

## 💻 实操

先跑 Verifier + Quality Gate，再跑 HITL 完整流程（创建 ticket → 人工决策 → 审计日志）。

In [14]:
class VerifierAgent:
    """对报告关键断言做 NLI 校验，输出 pass_rate"""
    name = 'verifier'

    def verify(self, report, job_id):
        claims = self._extract_claims(report)
        step_n = len(TRACE.get_job_trace(job_id)) + 1
        results = []
        for claim in claims:
            context   = random.choice(POLICY_CORPUS)
            supported = policy_nli_check(context, claim)
            results.append({'claim': claim[:50], 'supported': supported})
        pass_rate = sum(r['supported'] for r in results) / max(len(results), 1)
        verdict = {
            'pass':           pass_rate >= 0.80,
            'pass_rate':      round(pass_rate, 3),
            'claims_checked': len(results),
            'details':        results,
        }
        t = time.perf_counter()
        time.sleep(0.05 * len(claims))
        TRACE.record(job_id, self.name, step_n, 'report', verdict,
                     time.perf_counter() - t, extra={'pass_rate': pass_rate})
        return verdict

    def _extract_claims(self, report):
        """同时兼容 summary/risks 与 page1_summary/page2_risks 两种键名"""
        claims = []
        summary = report.get('summary') or report.get('page1_summary')
        if summary:
            claims.append(f"Summary: {summary[:40]}")
        risks = report.get('risks') or report.get('page2_risks') or []
        for r in risks[:3]:
            claims.append(r[:40])
        return claims or ['Content complies with company policy']


_verifier = VerifierAgent()
_sup      = Supervisor(min_pass_rate=0.80)

def orchestrate_with_verify(doc_id, force_pass=False, force_fail=False):
    """force_pass / force_fail 用于演示确定性的放行与拦截路径；
    都不传时使用真实 Verifier（NLI 结果为随机模拟）"""
    job_id = str(uuid.uuid4())
    _sup.start_job(job_id, doc_id)
    chunks   = policy_retrieve(f'policy {doc_id}', top_k=5, latency=0.2)
    summary  = policy_summarize(chunks, latency=0.2)
    risks    = policy_extract_risks(chunks, latency=0.2)
    measures = policy_propose_measures(risks, latency=0.15)
    TRACE.record(job_id, 'workers', 1, doc_id, 'done', 0.75)
    report = {'summary': summary, 'risks': risks, 'measures': measures}

    if force_fail:
        verdict = {'pass': False, 'pass_rate': 0.33, 'claims_checked': 3, 'details': []}
        TRACE.record(job_id, 'verifier', 2, 'report', verdict, 0.15,
                     extra={'pass_rate': 0.33})
        print('  [verifier          ] WARN pass_rate=0.33 (forced low quality)')
    elif force_pass:
        verdict = {'pass': True, 'pass_rate': 0.92, 'claims_checked': 3, 'details': []}
        TRACE.record(job_id, 'verifier', 2, 'report', verdict, 0.15,
                     extra={'pass_rate': 0.92})
        print('  [verifier          ] OK pass_rate=0.92 (forced high quality)')
    else:
        print('  [verifier          ]', end=' ')
        verdict = _verifier.verify(report, job_id)
        icon = 'OK' if verdict['pass'] else 'WARN'
        print(f"{icon} pass_rate={verdict['pass_rate']}  ({verdict['claims_checked']} claims)")

    gate = _sup.quality_gate(job_id, verdict['pass_rate'])
    print(f"  [quality_gate      ] action={gate['action']}  ({gate['reason']})")

    if gate['action'] == 'human_review':
        ticket = _sup.create_hitl_ticket(job_id, gate['reason'])
        _sup.close_job(job_id, 'under_review')
        return {'status': 'under_review', 'ticket_id': ticket, 'job_id': job_id}

    _sup.close_job(job_id, 'passed')
    return {'status': 'ok', 'report': report, 'verdict': verdict, 'job_id': job_id}


print('=== Scenario A: high quality (should proceed) ===')
r = orchestrate_with_verify('policy-2025', force_pass=True)
print(f"Result: {r['status']}\n")

print('=== Scenario B: low quality (should trigger HITL) ===')
r = orchestrate_with_verify('policy-2025', force_fail=True)
print(f"Result: {r['status']}  ticket={r.get('ticket_id')}\n")

print('=== Scenario C: real Verifier (random NLI results) ===')
r = orchestrate_with_verify('policy-2025')
print(f"Result: {r['status']}")

=== Scenario A: high quality (should proceed) ===
  [Supervisor] START job b82ee229
  [verifier          ] OK pass_rate=0.92 (forced high quality)
  [quality_gate      ] action=proceed  (pass_rate=0.92 >= 0.8)
  [Supervisor] CLOSE job b82ee229 -> passed
Result: ok

=== Scenario B: low quality (should trigger HITL) ===
  [Supervisor] START job d76d025a
  [verifier          ] WARN pass_rate=0.33 (forced low quality)
  [quality_gate      ] action=human_review  (pass_rate=0.33 < 0.8)
  [Supervisor] HITL ticket: TICKET-D76D02  (pass_rate=0.33 < 0.8)
  [Supervisor] CLOSE job d76d025a -> under_review
Result: under_review  ticket=TICKET-D76D02

=== Scenario C: real Verifier (random NLI results) ===
  [Supervisor] START job 01196645
  [verifier          ] OK pass_rate=1.0  (3 claims)
  [quality_gate      ] action=proceed  (pass_rate=1.00 >= 0.8)
  [Supervisor] CLOSE job 01196645 -> passed
Result: ok


In [10]:
import json
from enum import Enum


class HITLDecision(Enum):
    APPROVE = 'approve'
    REJECT  = 'reject'
    REVISE  = 'revise'

@dataclass
class HITLTicket:
    ticket_id:   str
    job_id:      str
    reason:      str
    report:      dict
    verdict:     dict
    status:      str               = 'pending'
    decision:    object            = None
    reviewer:    str               = ''
    reviewed_at: float             = 0.0
    notes:       str               = ''


class HITLSystem:
    def __init__(self):
        self.tickets = {}

    def create_ticket(self, job_id, report, verdict, reason):
        tid = f'TICKET-{job_id[:6].upper()}'
        t   = HITLTicket(tid, job_id, reason, report, verdict)
        self.tickets[tid] = t
        print(f'\n  HITL Ticket: {tid}')
        print(f'    reason:    {reason}')
        print(f"    pass_rate: {verdict.get('pass_rate', 'N/A')}")
        return t

    def review(self, ticket_id, decision, reviewer='reviewer@company.com', notes=''):
        t = self.tickets[ticket_id]
        t.status      = 'reviewed'
        t.decision    = decision
        t.reviewer    = reviewer
        t.reviewed_at = time.time()
        t.notes       = notes
        icons = {'approve': 'APPROVED', 'reject': 'REJECTED', 'revise': 'REVISE'}
        print(f"\n  [{icons[decision.value]}] {reviewer}: {decision.value}")
        if notes:
            print(f'    notes: {notes}')
        return t

    def audit_log(self):
        return [{
            'ticket_id':   t.ticket_id,
            'job_id':      t.job_id,
            'status':      t.status,
            'decision':    t.decision.value if t.decision else None,
            'reviewer':    t.reviewer,
            'pass_rate':   t.verdict.get('pass_rate'),
            'reviewed_at': round(t.reviewed_at, 2),
        } for t in self.tickets.values()]


hitl   = HITLSystem()
job_id = str(uuid.uuid4())

# 1. 系统触发 HITL（低质量）
low_verdict = {'pass': False, 'pass_rate': 0.50, 'claims_checked': 4, 'details': []}
ticket = hitl.create_ticket(job_id, {'summary': 'draft'}, low_verdict,
                             'pass_rate=0.50 < 0.80')

# 2. 人工审核：APPROVE
print('\n--- Human review ---')
hitl.review(ticket.ticket_id, HITLDecision.APPROVE,
            reviewer='alice@company.com',
            notes='Content acceptable; please improve verifier quality.')

# 3. 审计日志
print('\nAudit Log:')
print(json.dumps(hitl.audit_log(), ensure_ascii=False, indent=2))

# 4. REJECT 场景
job_id2 = str(uuid.uuid4())
bad = {'pass': False, 'pass_rate': 0.20, 'claims_checked': 5}
t2  = hitl.create_ticket(job_id2, {}, bad, 'pass_rate extremely low')
hitl.review(t2.ticket_id, HITLDecision.REJECT, reviewer='bob@company.com',
            notes='Report seriously incorrect, must regenerate.')
print(f'\nTotal audit entries: {len(hitl.audit_log())}')


  HITL Ticket: TICKET-2CFA13
    reason:    pass_rate=0.50 < 0.80
    pass_rate: 0.5

--- Human review ---

  [APPROVED] alice@company.com: approve
    notes: Content acceptable; please improve verifier quality.

Audit Log:
[
  {
    "ticket_id": "TICKET-2CFA13",
    "job_id": "2cfa1303-b4bb-4b0c-90c5-0a568590f781",
    "status": "reviewed",
    "decision": "approve",
    "reviewer": "alice@company.com",
    "pass_rate": 0.5,
    "reviewed_at": 1792232232.71
  }
]

  HITL Ticket: TICKET-44E695
    reason:    pass_rate extremely low
    pass_rate: 0.2

  [REJECTED] bob@company.com: reject
    notes: Report seriously incorrect, must regenerate.

Total audit entries: 2


# Part 16 · 实战：三页报告系统（逐层拆解）

## 系统需求

> **输入**：企业政策文档 ID
> **输出**：三页结构化报告
> - 第一页：要点总结
> - 第二页：风险清单
> - 第三页：建议措施
>
> **要求**：Verifier 校验关键断言；pass_rate < 0.8 触发 HITL；全程 trace；有输入校验

## 架构决策

| 组件 | 选型 | 理由 |
|------|------|------|
| Orchestrator-Driven | 静态计划 | 固定三任务，可预测、易测试 |
| `asyncio.gather` | 三个 Worker 并行 | 互相独立，总延迟取最大值 |
| Verifier + Quality Gate | NLI 校验 | 法务要求内容准确性 |
| HITL（REVIEW 级） | pass_rate < 0.8 阻塞 | 高风险决策人工兜底 |
| TraceStore | 轻量内存存储 | 教学场景，按 job_id 聚合 |

## 完整架构图

```mermaid
flowchart TB
    User["用户 POST /generate_report<br/>doc_id: policy-2025"]
    Supervisor["Supervisor<br/>start_job / close_job<br/>quality_gate / create_hitl_ticket"]
    Orchestrator["Orchestrator<br/>1. retriever.run<br/>2. asyncio.gather summarizer/risk/measure<br/>3. aggregate<br/>4. verifier.verify"]
    Retriever["Retriever 0.2s"]
    Summarizer["Summarizer 0.3s"]
    RiskExt["RiskExt 0.25s"]
    Measure["MeasureProposer 0.2s"]
    Verifier["Verifier NLI 校验"]

    User --> Supervisor
    Supervisor -->|监督| Orchestrator
    Orchestrator --> Retriever
    Orchestrator --> Summarizer
    Orchestrator --> RiskExt
    Orchestrator --> Measure
    Orchestrator --> Verifier
```

## Orchestrator 核心逻辑

```python
class ReportOrchestrator:
    def generate(self, doc_id: str) -> dict:
        job_id = str(uuid.uuid4())
        self.supervisor.start_job(job_id, doc_id)

        # Stage 1: 检索（其他阶段依赖此结果）
        ret = self._timed_run("retriever", self.retriever, doc_id, job_id, step=1)

        # Stage 2: 并行执行三个独立 Worker
        async def parallel():
            loop = asyncio.get_event_loop()
            async def w(name, worker, args, step):
                t = time.perf_counter()
                res = await loop.run_in_executor(None, worker.run, *args)
                TRACE.record(job_id, name, step, str(args)[:40], res, time.perf_counter() - t)
                return res
            return await asyncio.gather(
                w("summarizer", self.summarizer, (ret["chunks"],), step=2),
                w("risk_ext",   self.risk_ext,   (ret["chunks"],), step=3),
                w("measure",    self.measure,    (["r1", "r2"],),   step=4),
            )
        summ, risks, meas = asyncio.run(parallel())

        # Stage 3: 汇总 + Verify + Quality Gate
        report  = self._aggregate(summ, risks, meas)
        verdict = self.verifier.verify(report, job_id)
        gate    = self.supervisor.quality_gate(job_id, verdict["pass_rate"])
        return self._finalize(job_id, report, verdict, gate)
```

## Verifier 实现要点

```python
class VerifierAgent:
    """对报告关键断言做 NLI 校验"""

    def verify(self, report: dict, job_id: str) -> dict:
        # 1. 从报告中提取可验证断言
        claims = self._extract_claims(report)

        results = []
        for claim in claims:
            # 2. 检索支撑文本
            context = self._retrieve_support(claim)
            # 3. NLI 判断：context 是否蕴含 claim
            #    生产：cross-encoder/nli-deberta-v3-small 或 LLM-as-Judge
            supported = self._nli_entailment(context, claim)
            results.append({"claim": claim, "supported": supported})

        pass_rate = sum(r["supported"] for r in results) / max(len(results), 1)
        return {"pass": pass_rate >= 0.80, "pass_rate": round(pass_rate, 3), "details": results}
```

## 运行示例

**正常路径**（全部指标达标）

```
[Supervisor] START job a1b2c3d4
  [retriever         ] OK 0.201s   6 chunks
  [summarizer        ] OK 0.302s   (parallel)
  [risk_extractor    ] OK 0.251s   (parallel)
  [measure_proposer  ] OK 0.203s   (parallel)
  [verifier          ] OK pass_rate=0.917  (12 claims)
  [quality_gate      ] proceed  (pass_rate=0.917 >= 0.80)
[Supervisor] CLOSE job a1b2c3d4 -> passed

Trace [a1b2c3d4]  (5 steps)
────────────────────────────────────────────────────────
  step 1 | retriever          | ##          201ms
  step 2 | summarizer         | ###         302ms
  step 3 | risk_extractor     | ###         251ms
  step 4 | measure_proposer   | ##          203ms
  step 5 | verifier           | #           112ms
────────────────────────────────────────────────────────
  Total: 5 entries  wall clock: 0.609s  (vs serial 0.959s)
```

**并行节省**：`(0.302 + 0.251 + 0.203) - max(...) = 0.756 - 0.302 = 0.454s`

**HITL 触发路径**（pass_rate 不达标）

```
  [verifier          ] WARN pass_rate=0.583  (12 claims)
    ↳ failed claims:
       [2] "措施 3 与现行法规矛盾"   → NOT supported
       [5] "风险 2 已有解决方案"     → NOT supported
  [quality_gate      ] human_review  (pass_rate=0.583 < 0.80)
  [Supervisor] HITL ticket: TICKET-E5F6G7
[Supervisor] CLOSE job e5f6g7h8 -> under_review

返回给用户：{"status": "under_review", "ticket_id": "TICKET-E5F6G7"}
```

> 下面把前面所有组件组合起来跑一次端到端（Worker 延迟与 NLI 结果为模拟值，HITL 是否触发取决于随机 pass_rate）。

In [11]:
class ThreePageReportSystem:
    """Orchestrator + 4 Workers + Verifier + Supervisor + HITL 的端到端组合"""

    def __init__(self, min_pass_rate=0.80):
        self.retriever  = RetrieverWorker()
        self.summarizer = SummarizerWorker()
        self.risk_ext   = RiskExtractorWorker()
        self.measure    = MeasureProposerWorker()
        self.verifier   = VerifierAgent()
        self.supervisor = Supervisor(min_pass_rate=min_pass_rate)
        self.hitl       = HITLSystem()

    def generate_report(self, doc_id):
        job_id  = str(uuid.uuid4())
        t_total = time.perf_counter()
        self.supervisor.start_job(job_id, doc_id)

        # Stage 1: 检索（其他阶段依赖此结果）
        t   = time.perf_counter()
        ret = self.retriever.run(doc_id)
        TRACE.record(job_id, 'retriever', 1, doc_id, ret, time.perf_counter() - t)
        print(f"  [{'retriever':<18}] OK {round(time.perf_counter()-t, 3)}s  {ret['count']} chunks")

        # Stage 2: 三个独立 Worker 并行
        async def parallel():
            loop = asyncio.get_event_loop()
            async def w(worker, args, step):
                t = time.perf_counter()
                res = await loop.run_in_executor(None, worker.run, *args)
                lat = time.perf_counter() - t
                TRACE.record(job_id, worker.name, step, str(args)[:40], res, lat)
                print(f"  [{worker.name:<18}] OK {round(lat, 3)}s  (parallel)")
                return res
            return await asyncio.gather(
                w(self.summarizer, (ret['chunks'],),     step=2),
                w(self.risk_ext,   (ret['chunks'],),     step=3),
                w(self.measure,    (['riskA', 'riskB'],), step=4),
            )

        summ_r, risk_r, meas_r = asyncio.run(parallel())
        report = {
            'page1_summary':  summ_r['summary'],
            'page2_risks':    risk_r['risks'],
            'page3_measures': meas_r['measures'],
        }

        # Stage 3: Verifier 校验 + Quality Gate
        print(f'  [verifier          ]', end=' ')
        verdict = self.verifier.verify(report, job_id)
        icon = 'OK' if verdict['pass'] else 'WARN'
        print(f"{icon} pass_rate={verdict['pass_rate']}  ({verdict['claims_checked']} claims)")

        gate = self.supervisor.quality_gate(job_id, verdict['pass_rate'])
        print(f"  [quality_gate      ] {gate['action']}  ({gate['reason']})")

        # Stage 4: 放行 or 转人工
        if gate['action'] == 'human_review':
            ticket = self.hitl.create_ticket(job_id, report, verdict, gate['reason'])
            self.supervisor.close_job(job_id, 'under_review')
            TRACE.print_job_trace(job_id)
            return {'status': 'under_review', 'ticket_id': ticket.ticket_id,
                    'job_id': job_id, 'latency_s': round(time.perf_counter() - t_total, 3)}

        self.supervisor.close_job(job_id, 'passed')
        TRACE.print_job_trace(job_id)
        return {'status': 'ok', 'report': report, 'verdict': verdict,
                'job_id': job_id, 'latency_s': round(time.perf_counter() - t_total, 3)}


print('=' * 60)
print('Three-Page Report System - End-to-End Run')
print('=' * 60)
system = ThreePageReportSystem(min_pass_rate=0.80)
result = system.generate_report('policy-2025')

print(f"\nStatus:  {result['status']}")
print(f"Latency: {result['latency_s']}s")
if result['status'] == 'ok':
    r = result['report']
    print(f"\nPage 1 (Summary): {r['page1_summary'][:80]}")
    print('\nPage 2 (Risks):')
    for risk in r['page2_risks'][:3]:
        print(f'  - {risk[:65]}')
    print('\nPage 3 (Measures):')
    for m in r['page3_measures'][:3]:
        print(f'  - {m[:65]}')
else:
    print(f"需人工审核，ticket: {result['ticket_id']}")

Three-Page Report System - End-to-End Run
  [Supervisor] START job c196ef25
  [retriever         ] OK 0.201s  6 chunks
  [measure_proposer  ] OK 0.206s  (parallel)
  [risk_extractor    ] OK 0.255s  (parallel)
  [summarizer        ] OK 0.308s  (parallel)
  [verifier          ] OK pass_rate=1.0  (4 claims)
  [quality_gate      ] proceed  (pass_rate=1.00 >= 0.8)
  [Supervisor] CLOSE job c196ef25 -> passed

Trace [c196ef25]  (5 steps)
--------------------------------------------------------
  step  1 | retriever          | ##                 201ms
  step  4 | measure_proposer   | ##                 206ms
  step  3 | risk_extractor     | ###                255ms
  step  2 | summarizer         | ###                308ms
  step  5 | verifier           | ##                 205ms
--------------------------------------------------------
  Total: 5 entries, sum latency: 1175ms

Status:  ok
Latency: 0.933s

Page 1 (Summary): [Summary] Policy highlights: 公司差旅政策规定，国内出差需提前 3 天申请审批，国际出差需提前 10 天。... (6

# Part 17 · 最佳实践与常见坑

## 最佳实践 Top 10

| # | 实践 | 说明 |
|---|------|------|
| 1 | **Plan → Dispatch → Aggregate** | Orchestrator 三步法，不做具体任务 |
| 2 | **结构化消息 + JSON Schema** | 每个任务 / 结果都有 schema，接口稳定 |
| 3 | **幂等 + request_id** | 每条消息带唯一键，重试安全 |
| 4 | **每步必有 Trace** | job_id 串联，latency / tokens / model 全记 |
| 5 | **Quality Gate + 分级 HITL** | 自动校验，低置信度才转人工 |
| 6 | **并行化独立子任务** | `asyncio.gather` 减少总延迟 |
| 7 | **Worker 最小权限** | 每个 Worker 只有完成本职所需权限 |
| 8 | **模型分层路由** | Haiku / Sonnet / Opus 按任务复杂度选 |
| 9 | **重试 + 降级 + HITL** | 三级故障策略，不静默失败 |
| 10 | **渐进部署** | 串行 MVP → 并行 → 加 Verifier → 加 Supervisor |

## 常见坑速查表

| 坑 | 典型症状 | 根因 | 对策 |
|----|----------|------|------|
| Orchestrator 单点瓶颈 | Orch CPU 100%，Worker 空转 | 逻辑都堆在 Orch | 拆分调度，引入消息队列 |
| 无 trace 无法定位 | 出错后不知道哪步失败 | 懒得写 trace | 每步强制记录，不可跳过 |
| 没有幂等设计 | 重试导致重复写入 / 扣费 | 忘加 request_id | 消息级 + 操作级双幂等 |
| Agent 循环调用 | 资源耗尽，成本暴增 | A 调 B，B 调 A | 最大调用深度 + 停止条件 |
| 过早并发扩张 | 错误放大，成本飙升 | 没有串行 MVP | 先跑通再扩容 |
| 过度信任 LLM 输出 | 执行了危险 tool call | 没有输出校验 | 参数白名单 + HITL 审批 |
| 忽视数据隐私 | PII 在 Agent 间明文传播 | 缺少脱敏层 | Retriever 输出处脱敏 |
| Verifier 本身不可信 | Verifier 也会出错 | 信任 Verifier 100% | Verifier 也要 trace + 人工抽检 |

# Part 18 · 课堂练习

## 五阶段练习路径

| 阶段 | 任务 | 验收 |
|------|------|------|
| **1. MVP** | Orchestrator + 3 个串行 Worker，跑通端到端，有 trace 输出 | trace.json 可读 |
| **2. 并行** | Worker 改为 `asyncio.gather` 并行，测量延迟改善 | 延迟 < 串行 60% |
| **3. Verifier** | 加 Verifier + Quality Gate；pass_rate < 0.8 触发 HITL ticket | ticket 正确生成 |
| **4. Supervisor** | 完整 trace 存储 + p95 延迟监控 + Audit Log | trace 可按 job_id 查询 |
| **5. 安全** | Pydantic 入参校验 + 任务白名单 + PII 脱敏 | 恶意输入被拦截并记录 |

## 评估标准

```
功能正确      40%  ── 三页报告、Verifier、HITL 均工作
架构清晰      20%  ── 三层分离，接口有 schema
Trace/监控    15%  ── 每步有记录，有质量门
错误/降级     15%  ── 重试、降级、故障演练
文档与演示    10%  ── README + 架构图 + 演示录屏
```

## 推荐演示顺序

**从正常到异常，逐步展示系统能力：**

1. **正常路径**（pass_rate ≈ 0.9）
   - 展示完整 trace 甘特图
   - 指出并行节省的时间
2. **低质量触发 HITL**（pass_rate ≈ 0.6）
   - 展示 HITL ticket 内容
   - 展示审核界面（或 JSON 模拟）
3. **Worker 故障 → 降级**
   - 手动让 Reranker 抛异常
   - 展示降级路径和日志
4. **Trace 回放**
   - 从 trace.json 重现某次失败
   - 定位到具体的失败 step

## 工具栈补充

除「推荐资源」中列出的框架外，Verifier 与编排还有这些选择：

- **NLI / Verifier**：`cross-encoder/nli-deberta-v3-small`（轻量 NLI）、**RAGAS**（RAG 质量评估）、**TruLens**（LLM 应用评估）
- **生产工作流编排**：**Prefect**、**Temporal**（带持久化与重试的长任务）
- **向量数据库**：Qdrant / Weaviate（生产推荐）、Pinecone（Serverless）

## 总结

**Multi-Agent Architecture 的核心价值**

```
复杂任务 ──拆分──▶ 多个职责明确的 Agent ──协作──▶ 可扩展的生产系统
```

### 模式速查

| 主题 | 核心机制 | 适用场景 | Demo 实现 | 章节 |
|------|----------|----------|-----------|------|
| **Monolithic** | 串行函数调用 | 简单原型 | `monolithic_agent()` | Part 1 |
| **Orchestrator-Driven** | 静态 / 动态计划 + 中心编排 | 复杂工作流 | 见 Part 2 | Part 2 |
| **Peer-to-Peer** | 竞价 / 协商 | 自治系统、多方协商 | 见 Part 2 | Part 2 |
| **Pipeline** | `queue.Queue` + 线程 Worker | 线性流水、高并发 | `q_retrieve / q_rerank / q_generate` | Part 3 |
| **Hub-and-Spoke** | `asyncio.gather` 并行 | 多角色协作、并行加速 | `orchestrator()` | Part 4 |
| **Blackboard** | 共享字典 + `threading.Event` | 迭代协作、松耦合 | `Blackboard` 类 | Part 5 |
| **通信机制** | Sync RPC / 队列 / Pub-Sub / 共享存储 | 决定 Agent 之间怎么通信 | 决策树 | Part 6 |
| **幂等** | `request_id` + 原子占位 + DB 唯一索引 | 所有消息驱动场景 | `IdempotentProcessor` / 生产版伪代码 | Part 7 |
| **Backpressure** | 有界队列 + 阻塞 / 拒绝 | 生产者快于消费者 | 有界队列 Demo | Part 8 |
| **可观测性** | `span()` + `Metrics` | 生产监控、性能诊断 | `traced_pipeline()` | Part 9 |
| **安全隔离** | 最小权限 + 入参校验 + 白名单 | 对外暴露的 Agent | `TaskRequest` / `PERMISSIONS` | Part 10 |
| **成本优化** | 分层路由 / 缓存 / 批处理 | LLM 调用量大 | 路由 + 缓存 | Part 11 |
| **优雅降级** | `try/except` + 降级路径 | 高可用系统 | `ResilientOrchestrator` | Part 12 |
| **分层架构** | Worker / Orchestrator / Supervisor | 生产级 Agentic 系统 | `SimpleOrchestrator` + `Supervisor` | Part 14 |
| **Agent Supervision** | Verifier + Quality Gate + HITL + Audit | 合规、高风险决策 | `VerifierAgent` / `HITLSystem` | Part 15 |
| **端到端实战** | 分层 + 并行 + 监督组合 | 三页报告系统 | `ThreePageReportSystem` | Part 16 |
| **最佳实践** | Top 10 实践 + 坑速查 | 落地前自查 | — | Part 17 |

### 三条铁律

1. **每条消息带 `request_id`**，处理前先做幂等检查
2. **每个 Agent 暴露 latency + status**，不监控等于盲飞
3. **提前设计失败路径**，不要等故障发生后再想降级逻辑

### 分层架构的三条铁律

1. **每层只做自己的事** — Worker 不计划，Orchestrator 不检索
2. **每步必有 trace** — 无 trace = 盲飞，出事无法定位
3. **质量门 + HITL** — 自动校验 + 人工兜底，不纯自动化高风险决策

### 技术演进路径

```
单体 Agent
  → Pipeline（Redis Queue）
    → Hub-and-Spoke（HTTP 微服务）
      → Blackboard（事件驱动）
        → 分层架构 + Supervision（Worker / Orchestrator / Supervisor）
          → 混合架构（按需组合）
```

### 渐进落地路线

```mermaid
flowchart LR
    A[单体 MVP] --> B[串行三层架构<br/>有 trace]
    B --> C[并行 Workers<br/>asyncio.gather]
    C --> D[Verifier + Quality Gate]
    D --> E[Supervisor + HITL]
    E --> F[容器化 + 压测 + 监控]
```

---
*下一步：把这些 Agent 换成真实的 LLM API 调用，并接入 Redis / Kafka 消息队列*

## 推荐资源

### 框架 & 工具
- **LangChain / LangGraph** — Multi-Agent patterns
- **CrewAI** — Role-based agents
- **AutoGen** — Microsoft multi-agent framework
- **Haystack** — Production RAG pipelines

### 消息队列
- **Redis Streams** — 入门首选
- **RabbitMQ** — 经典 AMQP
- **Apache Kafka** — 高吞吐场景

### 可观测性
- **LangSmith / Langfuse** — LLM tracing
- **OpenTelemetry** — 标准 traces
- **Prometheus + Grafana** — Metrics

### 部署
- **Docker Compose** — 本地多服务
- **Kubernetes + HPA** — 生产弹性扩缩
- **Pinecone / Qdrant / OpenSearch** — 向量数据库

### 评估
- **RAGAS** — RAG 质量评估框架